# S3 · RoBERTa + BGE-M3 frozen → fitur gabungan → Ridge

**Tiga input tetap:** soal, jawaban referensi, respons siswa. Target satu `rater` 0–100.
Ini **feature fusion**, bukan penggabungan dua skor. Resep vektor mengikuti E21:

**RoBERTa CLS768 → LayerNorm** + **BGE frozen1024 → proyeksi256 → LayerNorm** → **concat1024 → Ridge**.

Representasi dilatih dengan MLP bantu hanya pada train, kemudian dikunci dan head diganti Ridge. MLP dari checkpoint yang sama tetap dievaluasi sebagai kontrol. S2 BGE frozen→Ridge alpha0,01 menjadi baseline tambahan, bukan hasil lama yang disalin.

Pilih **T4 GPU → Run all → unggah `fix_dataset_300.xlsx`**. Tanpa Drive atau clone repo. Ada16pelatihan representasi pada konfigurasi5fold:5outer+10pasangan inner+1final. Ini lebih berat dari notebook S2; waktu sebenarnya bergantung GPU/panjang respons. Cache tahap selesai dan progres epoch bersifat lokal; unduh ZIP sebelum runtime hilang.

**Belum ada hasil penelitian dari notebook ini.** Arsitektur ini adalah hipotesis yang diuji, bukan kombinasi yang sudah dinyatakan optimal oleh paper.

## 1 · Setup dan upload

Semua kode diembed. Worker numerik terpisah mempertahankan Torch CUDA Colab. Data, cache dan checkpoint berada di disk runtime lokal. Hasil tidak diunggah otomatis ke GitHub.

In [ ]:
import os, sys, json, subprocess, html, secrets, atexit, argparse, hashlib, venv
from pathlib import Path
from IPython.display import display, HTML
try:
    from packaging.requirements import Requirement
except ImportError:
    from pip._vendor.packaging.requirements import Requirement
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
default_workspace='/content/AES_300_S3_Fusion_Ridge_Tiga_Input_Colab' if IN_COLAB else str(Path.cwd()/'outputs/AES_300_S3_Fusion_Ridge_Tiga_Input_Colab')
ROOT=Path(os.environ.get('AES300_WORKSPACE',default_workspace)).expanduser().absolute()
ROOT=ROOT.resolve()
if ROOT==Path('/content/drive') or Path('/content/drive') in ROOT.parents:
    raise ValueError('Workspace harus lokal; S3 fusion Ridge tidak menulis Google Drive.')
ROOT.mkdir(parents=True,exist_ok=True)
def prepare_local_storage(workspace, in_colab):
    storage=Path(workspace).expanduser().resolve()
    if storage==Path('/content/drive') or Path('/content/drive') in storage.parents:
        raise ValueError('S3 fusion Ridge hanya menyimpan lokal; pilih workspace di luar Google Drive.')
    storage.mkdir(parents=True,exist_ok=True)
    probe=storage/'.aes_write_check';probe.write_text('local write check');probe.unlink()
    return storage

# Identitas kode notebook untuk membedakan hasil dari implementasi yang berbeda.
NOTEBOOK_CODE_SHA256='ec58fc4a613a830ca27a69a820ca227f70090a48b87440aeb255934812a66a7c'
print('Ruang kerja AES siap. Seluruh kode tersedia di notebook ini.')
print('Versi Python:',sys.version.split()[0])

# Hanya pustaka runtime yang diperlukan; alat pengujian tidak dipasang untuk menjalankan AES.
REQUIREMENTS_TEXT=r'''
# Torch dipertahankan dari runtime Colab supaya CUDA tidak diganti wheel CPU.
# Tanpa torch, setup memasang torch 2.6.0 CPU sebelum file ini.
numpy==2.2.6
pandas==2.2.3
scipy==1.15.3
scikit-learn==1.6.1
# tokenizers0.19 (transformers4.44) tidak menyediakan wheel Python3.13.
transformers==4.44.2; python_version == '3.12'
transformers==4.48.3; python_version == '3.13'
openpyxl==3.1.5
'''
(ROOT/'requirements.dataset300.txt').write_text(REQUIREMENTS_TEXT)

def resolve_dataset_upload(workspace, configured='', search_paths=None, browser_upload=False):
    """Web Colab: pakai file yang ada, atau tampilkan Choose Files pada cell ini."""
    from pathlib import Path
    import hashlib
    import os
    import shutil
    workspace=Path(workspace).expanduser().resolve()
    destination=workspace/'data/dataset300/fix_dataset_300.xlsx'
    if configured:
        candidates=[Path(configured).expanduser().resolve()]
        if not candidates[0].is_file():
            raise FileNotFoundError('DATASET_FILE/AES300_DATASET belum ditemukan: '+str(candidates[0]))
    else:
        places=search_paths if search_paths is not None else [Path('/content'),Path.cwd(),workspace]
        candidates=[]
        for place in places:
            path=(Path(place)/'fix_dataset_300.xlsx').resolve()
            if path.is_file() and path not in candidates and path!=destination:
                candidates.append(path)
        if not candidates and destination.is_file():
            candidates=[destination]
    if not candidates and browser_upload:
        from google.colab import files
        destination.parent.mkdir(parents=True,exist_ok=True)
        print('Klik Choose Files, lalu pilih fix_dataset_300.xlsx dari komputer.',flush=True)
        previous=Path.cwd()
        try:
            os.chdir(destination.parent)
            uploaded=files.upload()
        finally:
            os.chdir(previous)
        if 'fix_dataset_300.xlsx' not in uploaded:
            raise ValueError('Pilih fix_dataset_300.xlsx. Jika dibatalkan, jalankan ulang cell setup ini.')
        candidates=[destination]
        del uploaded
    if not candidates:
        raise FileNotFoundError(
            'Dataset belum ada. Di web Colab, jalankan cell setup ini dan pilih '
            'fix_dataset_300.xlsx melalui Choose Files. Alternatif: panel Files/Berkas kiri '
            '→ Upload, lalu jalankan ulang cell ini. Jika lokasinya berbeda, isi DATASET_FILE.')
    hashes={hashlib.sha256(p.read_bytes()).hexdigest() for p in candidates}
    if len(hashes)>1:
        raise ValueError('Ada beberapa dataset berbeda: '+', '.join(map(str,candidates))+
                         '. Isi DATASET_FILE dengan path file yang ingin dipakai.')
    source=candidates[0]
    if not source.stat().st_size:
        raise ValueError('File upload masih kosong; tunggu upload selesai lalu jalankan ulang cell ini.')
    destination.parent.mkdir(parents=True,exist_ok=True)
    if source!=destination:
        shutil.copy2(source,destination)
    print('Dataset dari upload lokal:',source,'→',destination,flush=True)
    return destination


In [ ]:
# Pemeriksaan dan pemasangan pustaka tersedia langsung di sini.
INSPECT = r'''
import json, sys
from importlib import metadata, util
try:
    from packaging.requirements import Requirement
except ImportError:
    from pip._vendor.packaging.requirements import Requirement
roots=json.loads(sys.argv[1])+['torch>=2.6.0']
versions={}
for raw in roots:
    name=Requirement(raw).name
    try:versions[name]=metadata.version(name)
    except metadata.PackageNotFoundError:versions[name]=None
queue=[Requirement(raw) for raw in roots];seen=set();issues=[];install_requirements=[]
while queue:
    req=queue.pop()
    if req.marker and not req.marker.evaluate({'extra':''}):continue
    try:dist=metadata.distribution(req.name)
    except metadata.PackageNotFoundError:
        issues.append(str(req)+' belum tersedia');install_requirements.append(str(req));continue
    if not req.specifier.contains(dist.version,prereleases=True):
        issues.append(str(req)+' diperlukan; tersedia '+dist.version);install_requirements.append(str(req))
    name=dist.metadata['Name'].lower().replace('_','-')
    if name in seen:continue
    seen.add(name)
    queue.extend(Requirement(raw) for raw in (dist.requires or []))
print(json.dumps({'python':sys.version.split()[0],'runtime_version':list(sys.version_info[:2]),
 'versions':versions,'torch_present':util.find_spec('torch') is not None,
 'dependency_issues':sorted(set(issues)),'install_requirements':sorted(set(install_requirements)),'dependency_count':len(seen)}))
'''

IMPORTS = r'''
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung pustaka AES.
    # Hanya worker AES yang diubah; paket Colab dan operator torch tetap utuh.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
import json
import torch, numpy, pandas, scipy, sklearn, openpyxl
from transformers import AutoModel, AutoTokenizer
available=torch.cuda.is_available()
print(json.dumps({'torch':torch.__version__,'cuda_available':available,
 'device':torch.cuda.get_device_name(0) if available else 'CPU',
 'checkpoint_loaded':False,'imports_verified':True,'transformers_backend':'pytorch_text_only'}))
'''

def environment_folder(root, version):
    version=tuple(version[:2])
    if version not in ((3,12),(3,13)):
        raise RuntimeError('Setup AES mendukung Python 3.12 atau 3.13. Versi lain belum diuji.')
    return Path(root)/'outputs'/('dataset300_env_text_e6_gpu_reuse' if version==(3,12) else 'dataset300_env_py313_text_e6_gpu_reuse')


# Pilih versi pustaka sesuai Python aktif agar aturan Python 3.12 dan 3.13 tidak tertukar.
def active_requirements(path, version):
    """Hanya aturan untuk versi Python aktif; tidak mencoba semua versi Transformers."""
    selected=[]
    for line in Path(path).read_text().splitlines():
        line=line.split('#',1)[0].strip()
        if not line:continue
        req=Requirement(line)
        if req.marker and not req.marker.evaluate({'python_version':'.'.join(map(str,version[:2])), 'extra':''}):continue
        selected.append(req.name+str(req.specifier))
    return selected


class SetupError(RuntimeError):
    pass


# Memeriksa pustaka yang tersedia, menyiapkan versi yang diperlukan, lalu memvalidasi import sebelum melanjutkan.
def prepare_environment(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--verify-python',help='Periksa worker yang disediakan pengguna tanpa memasang paket.')
    args=parser.parse_args(argv)
    folder=environment_folder(ROOT,sys.version_info)
    outputs=ROOT/'outputs';outputs.mkdir(parents=True,exist_ok=True)
    ready=outputs/'dataset300_setup.json';ready.unlink(missing_ok=True)
    logfile=outputs/'dataset300_setup.log'
    requirements=active_requirements(ROOT/'requirements.dataset300.txt',sys.version_info)
    env={**os.environ,'HF_HOME':str(outputs/'cache/huggingface'),'PIP_DISABLE_PIP_VERSION_CHECK':'1'}
    with logfile.open('w') as log:
        def execute(command,stage):
            log.write(stage+'\n'+json.dumps(command)+'\n');log.flush()
            process=subprocess.run(command,cwd=ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
            log.write(process.stdout or '');log.write(process.stderr or '');log.flush()
            if process.returncode:
                details='\n'.join((process.stderr or process.stdout or 'Proses gagal tanpa rincian.').splitlines()[-18:])
                raise SetupError(stage+' gagal.\n'+details+'\nRincian lengkap: '+str(logfile))
            return process.stdout
        def inspect(python):
            result=json.loads(execute([python,'-c',INSPECT,json.dumps(requirements)],'Pemeriksaan versi dan dependensi'))
            if result['runtime_version']!=list(sys.version_info[:2]):
                raise SetupError('Versi Python worker berbeda dari runtime. Gunakan lingkungan untuk versi runtime ini.')
            return result

        print('1/3 · Memeriksa lingkungan dan versi pustaka AES...',flush=True)
        if args.verify_python:
            # abspath menjaga symlink bin/python venv; resolve akan memakai Python global.
            python=str(Path(args.verify_python).absolute())
        else:
            if folder.exists() and not (folder/'pyvenv.cfg').is_file():
                raise SetupError('Folder target bukan lingkungan Python AES; tidak ditimpa.')
            if not (folder/'bin/python').exists():
                # Torch CUDA tetap berasal dari runtime. Paket AES yang berbeda dipasang lokal.
                venv.EnvBuilder(with_pip=False,system_site_packages=True).create(folder)
            python=str(folder/'bin/python')
        before=inspect(python)
        torch_version=before['versions'].get('torch')
        if torch_version and not Requirement('torch>=2.6.0').specifier.contains(torch_version,prereleases=True):
            raise SetupError('PyTorch minimal 2.6.0 diperlukan. Pilih runtime Colab yang lebih baru; torch bawaan tidak diganti.')
        if not before['torch_present']:
            raise SetupError('PyTorch bawaan runtime belum tersedia. Pilih runtime Colab yang baru; setup ini tidak memasang wheel CPU.')
        installed=False
        after=before
        if after['dependency_issues']:
            if args.verify_python:
                raise SetupError('Worker yang dipilih belum memenuhi kebutuhan AES. Rincian: '+str(logfile))
            print('2/3 · Menyiapkan pustaka AES tanpa mengganti PyTorch runtime...',flush=True)
            # --no-deps mencegah pustaka AES menarik ulang torch. Dependensi lain
            # diperiksa ulang dari metadata dan dipasang hanya jika diperlukan.
            pip=[sys.executable,'-m','pip','--python',python,'install','--ignore-installed','--no-deps','--no-input']
            for attempt in range(8):
                if not after['dependency_issues']:
                    break
                pending=after['install_requirements']
                if not pending or any(Requirement(raw).name.lower().replace('_','-')=='torch' for raw in pending):
                    raise SetupError('Dependensi meminta perubahan PyTorch. Gunakan runtime Colab yang kompatibel; torch bawaan tidak ditimpa.')
                execute(pip+pending,'Pemasangan pustaka AES privat, putaran '+str(attempt+1))
                installed=True
                after=inspect(python)
                if after['versions'].get('torch')!=torch_version:
                    raise SetupError('Versi PyTorch berubah selama setup; training tidak dimulai.')
        else:
            print('2/3 · Versi pustaka sudah cocok; tidak perlu instalasi ulang.',flush=True)
        if after['dependency_issues']:
            raise SetupError('Dependensi AES belum cocok: '+'; '.join(after['dependency_issues'][:3])+'. Rincian: '+str(logfile))
        # Jangan mengganti torch yang terpasang tetapi rusak dengan wheel CPU secara diam-diam.
        print('3/3 · Memeriksa import dan perangkat perhitungan...',flush=True)
        checked=json.loads(execute([python,'-c',IMPORTS],'Pemeriksaan import AES dan PyTorch'))
        result={**after,**checked,'python_version':after['python'],'python':python,
                'packages_action':'installed' if installed else 'reused','torch_reused':before['torch_present'],
                'log_file':str(logfile),'requirements_sha256':hashlib.sha256((ROOT/'requirements.dataset300.txt').read_bytes()).hexdigest()}
        ready.write_text(json.dumps(result,indent=2)+'\n')
    print('Persiapan AES selesai. Versi, dependensi, dan import sudah diperiksa.',flush=True)
    return result


In [ ]:
# Penghubung proses Python: menjaga variabel antar-cell %%aes tanpa file pendukung proyek.
WORKER_SOURCE=r'''
import json, sys, traceback, types
from pathlib import Path

prefix, workspace = sys.argv[1:3]
module = types.ModuleType('__aes_notebook__')
sys.modules[module.__name__] = module
namespace = module.__dict__
namespace['ROOT'] = Path(workspace)

def event(kind, **values):
    print(prefix+json.dumps({'kind':kind, **values},ensure_ascii=False,allow_nan=False),flush=True)

def show(frame, title='', rows=8, columns=8):
    # Fungsi tampilan diisi oleh cell notebook; tidak diimpor dari file proyek.
    import pandas as pd
    frame = frame if isinstance(frame,pd.DataFrame) else pd.DataFrame(frame)
    view = namespace['preview_table'](frame.columns,frame.head(rows).to_dict('records'),title,
                                    row_count=len(frame),max_rows=rows,max_columns=columns)
    view['rows'] = json.loads(pd.DataFrame(view['rows']).to_json(orient='values',force_ascii=False,default_handler=str))
    event('table',**view,html=namespace['render_table'](view))

namespace['show'] = show
event('ready',python=sys.version.split()[0])
for line in sys.stdin:
    try:
        payload = json.loads(line)
        if payload.get('command') == 'close':break
        exec(compile(payload['code'],payload.get('filename','<aes-cell>'),'exec'),namespace)
    except BaseException:
        event('error',traceback=traceback.format_exc())
    else:
        event('done')
'''


In [ ]:
DATASET_FILE=''  # Biasanya otomatis dari /content; isi path lengkap jika berbeda.
PYTHON = os.environ.get('AES300_PYTHON','')
def show_installation_log():
    path=ROOT/'outputs/dataset300_setup.log'
    contents=path.read_text() if path.is_file() else 'Log belum tersedia.'
    display(HTML('<details style="background:#f8fafc;color:#172035;padding:12px;border:1px solid #94a3b8">'
                 '<summary>Rincian instalasi — buka jika diperlukan</summary>'
                 '<pre style="max-height:320px;overflow:auto;white-space:pre-wrap;color:#172035">'
                 +html.escape(contents)+'</pre></details>'))
try:
    ready=prepare_environment(['--verify-python',PYTHON] if PYTHON else [])
except Exception:
    show_installation_log()
    raise
if ready['runtime_version'] != list(sys.version_info[:2]) or not ready['imports_verified']:
    raise RuntimeError('Metadata setup tidak cocok runtime aktif.')
PYTHON=ready['python']
setup_rows=[
    ('Status','Siap — versi, dependensi, dan import AES lulus pemeriksaan'),
    ('Python',ready['python_version']),
    ('PyTorch',ready['torch']),
    ('Perangkat','RoBERTa/BGE: GPU; Ridge: CPU float64'),
    ('Pustaka AES','Versi yang cocok dipakai ulang' if ready['packages_action']=='reused'
                    else 'Versi khusus AES dipasang; pustaka bawaan tetap disimpan'),
    ('Pustaka utama','NumPy '+ready['versions']['numpy']+'; Transformers '+ready['versions']['transformers']),
    ('Model','Belum dimuat; tahap ini hanya menyiapkan lingkungan')]
status_html=''.join('<tr><th style="padding:9px;text-align:left;border:1px solid #94a3b8">'+html.escape(label)
                    +'</th><td style="padding:9px;border:1px solid #94a3b8">'+html.escape(value)+'</td></tr>'
                    for label,value in setup_rows)
display(HTML('<table style="background:#f8fafc!important;color:#172035!important;border-collapse:collapse">'
             +status_html+'</table>'))
show_installation_log()
PERSISTENT_ROOT=prepare_local_storage(ROOT,IN_COLAB)
print('Penyimpanan data, checkpoint dan Excel:',PERSISTENT_ROOT)
print('Checkpoint hanya lokal. Bisa dilanjutkan selama folder runtime masih ada; runtime dihapus berarti file lokal hilang. Unduh hasil sebelum menutup sesi.')
# Web Colab biasa: Choose Files muncul jika dataset belum diunggah.
DATA_PATH=resolve_dataset_upload(PERSISTENT_ROOT,DATASET_FILE or os.environ.get('AES300_DATASET',''),browser_upload=IN_COLAB)
def display_aes_table(event):
    display(HTML(event['html']))
if '_aes_process' in globals() and _aes_process.poll() is None:
    _aes_process.terminate()
_aes_prefix='__AES_'+secrets.token_hex(12)+'__'
child_env={**os.environ,'HF_HOME':str(ROOT/'outputs/cache/huggingface'),
           'AES300_DATASET':str(DATA_PATH),'AES300_RESULTS':str(PERSISTENT_ROOT/'outputs/dataset300_inline'),
           'AES300_REQUIRE_GPU':'0',
           'TOKENIZERS_PARALLELISM':'false',
           'OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','PYTHONUNBUFFERED':'1'}
child_env.pop('PYTHONPATH',None)
_aes_process=subprocess.Popen([PYTHON,'-u','-c',WORKER_SOURCE,_aes_prefix,str(ROOT)],
                              cwd=ROOT,env=child_env,stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1)
while True:
    line=_aes_process.stdout.readline()
    if not line:
        raise RuntimeError('Worker gagal mulai; lihat log setup.')
    if line.startswith(_aes_prefix):
        print('Proses perhitungan siap. Python',json.loads(line[len(_aes_prefix):])['python']);break
    print(line,end='')
def aes_cell(line,cell):
    if _aes_process.poll() is not None:
        raise RuntimeError('Worker berhenti; jalankan setup dan cell berikutnya dari awal.')
    _aes_process.stdin.write(json.dumps({'code':cell,'filename':'<notebook-cell>'})+'\n')
    _aes_process.stdin.flush()
    while True:
        output=_aes_process.stdout.readline()
        if not output:
            raise RuntimeError('Worker terhenti; jika RAM penuh pilih runtime lebih besar.')
        if not output.startswith(_aes_prefix):
            print(output,end='',flush=True);continue
        event=json.loads(output[len(_aes_prefix):])
        if event['kind']=='done':break
        if event['kind']=='error':raise RuntimeError(event['traceback'])
        if event['kind']=='table':display_aes_table(event)
get_ipython().register_magic_function(aes_cell,'cell','aes')
atexit.register(lambda:_aes_process.terminate() if _aes_process.poll() is None else None)
print('Dataset siap. Jalankan cell %%aes berurutan.')


## 2 · Konfigurasi sebelum melihat hasil

LR encoder5e-6; proyeksi/norm/MLP1e-4; AdamW decay0,01; batch efektif8/micro1; dropout0,1; warmup10%; clipping1. Resep diwarisi E21 sebagai kontrol, bukan parameter optimum dari paper.

**25epoch tetap**, tanpa early stopping memakai outer validation. Ini berbeda dari run historis E21, sehingga kontrol yang paling sepadan adalah MLP yang dilatih di notebook ini. Ridge tidak memiliki LR/epoch; grid alpha pada fitur concat mentah adalah0,01/0,1/1/10/100. Skala fitur fusion berbeda dari BGE L2, jadi alpha S2 tidak langsung dipindahkan sebagai alpha optimal S3.

In [ ]:
%%aes
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging, html
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung PEFT.
    # Hanya worker AES yang diubah; paket Colab dan operator torch tetap utuh.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

NOTEBOOK_CODE_SHA256='ec58fc4a613a830ca27a69a820ca227f70090a48b87440aeb255934812a66a7c'
SCOPE='s3_fusion_ridge_nested_cv'
REQUIRED=['Email address','response_number','response_text','soal','jawaban_referensi','rater']

In [ ]:
%%aes
# Ubah parameter sebelum run; jangan memilih dari test historis.
EPOCHS, FOLDS = 25, 5
S3_RIDGE_ALPHAS = [.01, .1, 1., 10., 100.]
S2_RIDGE_ALPHA = .01
DEVICE = 'cuda'
EXPECTED_ROWS = 300
cfg = dict(dataset=os.environ['AES300_DATASET'], output=str(ROOT/'outputs/s3_fusion_ridge'),
    sheet='Sheet1', expected_rows=EXPECTED_ROWS, score_min=0, score_max=100,
    test_fraction=.2, folds=FOLDS, split_seed=2026, split_candidates=128, seed=42,
    epochs=EPOCHS, early_stopping_patience=5, effective_batch=8, micro_batch=1,
    s3_encoder_lr=5e-6, encoder_lr=5e-6, projection_lr=1e-4, head_lr=1e-4,
    weight_decay=.01, warmup_ratio=.1, gradient_clip=1., dropout=.1,
    roberta_max_length=512, roberta_overlap=128, bge_max_length=1024, bge_overlap=128, bge_batch=4,
    device=DEVICE, context_max_tokens=192, question_max_tokens=64, input_format='single_sequence',
    s3_ridge_alphas=S3_RIDGE_ALPHAS, s2_ridge_alpha=S2_RIDGE_ALPHA,
    representation_policy='E21_fusion_fixed_epochs_train_only', code_sha256=NOTEBOOK_CODE_SHA256,
    roberta=dict(checkpoint='flax-community/indonesian-roberta-base', revision='cc5ecee7de8532880ad08d802292a6f42865bc83'),
    bge=dict(checkpoint='BAAI/bge-m3', revision='5617a9f61b028005a4858fdac845db406aefb181'))
if FOLDS < 3 or EPOCHS < 1:
    raise ValueError('Minimal3fold dan epoch positif.')
if DEVICE=='cuda' and not torch.cuda.is_available():
    raise RuntimeError('Pilih Runtime → Change runtime type → T4 GPU, lalu jalankan ulang.')
torch.set_num_threads(2)
print('Pelatihan representasi:', FOLDS+FOLDS*(FOLDS-1)//2+1, '| epoch setiap fit:', EPOCHS, flush=True)
print('Semua3teks → feature fusion → Ridge; kontrol MLP pada fitur sama.', flush=True)

## 3 · Dataset asli

Header: `Email address`, `response_number`, `response_text`, `soal`, `jawaban_referensi`, `rater`. Email/duplikat menjadi kelompok pencegah kebocoran dan tidak menjadi fitur model. Label tidak diisi, dibulatkan atau diganti otomatis.

In [ ]:
%%aes
def clean(text, normalize_punctuation=False):
    text = unicodedata.normalize('NFC', text).replace('\r\n','\n').replace('\r','\n')
    text = ''.join(c for c in text if c in '\n\t' or unicodedata.category(c) != 'Cc')
    if normalize_punctuation:
        text = text.translate(str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-'})).replace('…','...')
    return text


def digest(path):
    # Checkpoint besar diperiksa bertahap supaya tidak dimuat seluruhnya ke RAM.
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


@dataclass
class Bundle:
    rows: list
    y: np.ndarray
    groups: np.ndarray
    source_sha256: str
    summary: dict

    @property
    def texts(self):
        return [r['text'] for r in self.rows]


def read_dataset(cfg):
    path = Path(cfg['dataset'])
    source_hash = digest(path)
    book = load_workbook(path, read_only=True, data_only=True)
    try:
        if cfg['sheet'] not in book.sheetnames:
            raise ValueError(f"Tab {cfg['sheet']} tidak tersedia: {book.sheetnames}")
        data = iter(book[cfg['sheet']].iter_rows(values_only=True))
        header = [str(x).strip() if x is not None else None for x in next(data)]
        if any(header.count(x) != 1 for x in REQUIRED):
            raise ValueError('Enam header wajib hilang atau duplikat: '+', '.join(REQUIRED))
        rows = []
        for excel_row, values in enumerate(data, 2):
            if all(v is None for v in values):
                continue  # Hanya baris format kosong, bukan label/jawaban yang hilang.
            raw = {k: values[i] for i, k in enumerate(header) if k is not None}
            if any(raw[k] is None or not str(raw[k]).strip() for k in REQUIRED):
                raise ValueError(f'Baris {excel_row}: kolom wajib kosong: '+', '.join(k for k in REQUIRED if raw[k] is None)+'; tidak diisi otomatis.')
            score = raw['rater']
            if isinstance(score, bool) or not isinstance(score, (int, float)) or not math.isfinite(score):
                raise ValueError(f'Baris {excel_row}: rater harus angka skor guru, bukan teks/ID.')
            if int(score) != score or not cfg['score_min'] <= score <= cfg['score_max']:
                raise ValueError(f'Baris {excel_row}: skor bukan kategori integer dalam rentang yang dinyatakan.')
            for key in ('soal','jawaban_referensi'):
                if not isinstance(raw[key],str) or not raw[key].strip():
                    raise ValueError(f'Baris {excel_row}: {key} harus teks yang tidak kosong.')
            if not isinstance(raw['response_text'], str):
                raise ValueError(f'Baris {excel_row}: jawaban bukan teks.')
            rows.append({'row_id': f'row_{excel_row-1:04d}', 'source_row': excel_row,
                         'text': clean(raw['response_text']), 'raw_text': raw['response_text'],
                         'email_group': unicodedata.normalize('NFC', str(raw['Email address'])).strip().casefold(),
                         'question': clean(raw['soal']), 'reference': clean(raw['jawaban_referensi']),
                         'response_number': str(raw['response_number']), 'score': int(score)})
    finally:
        book.close()
    if cfg.get('expected_rows') is not None and len(rows) != cfg['expected_rows']:
        raise ValueError(f"Diharapkan {cfg['expected_rows']} baris, ditemukan {len(rows)}; tidak dipotong otomatis.")
    if not rows:
        raise ValueError('Dataset kosong')
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    email_first, duplicate_sets = {}, {}
    for i, row in enumerate(rows):
        email = row['email_group']
        if email in email_first:
            union(email_first[email], i)
        email_first[email] = i
        # Normalisasi ini HANYA audit kesamaan, bukan teks masukan encoder.
        normalized = ' '.join(row['text'].casefold().split())
        key = hashlib.sha256(normalized.encode()).hexdigest()
        duplicate_sets.setdefault(key, []).append(i)
    for indices in duplicate_sets.values():
        for i in indices[1:]:
            union(indices[0], i)
    roots = {root: i for i, root in enumerate(sorted({find(i) for i in range(len(rows))}))}
    groups = np.asarray([roots[find(i)] for i in range(len(rows))], dtype=int)
    y = np.asarray([r['score'] for r in rows], dtype=float)
    summary = {'scope': SCOPE, 'rows': len(rows), 'target_columns': ['rater'], 'model_outputs': 1,
               'score_range_observed': [float(y.min()), float(y.max())],
               'score_range_declared': [cfg['score_min'], cfg['score_max']],
               'scale_basis': '0–100 mengikuti notebook unggahan; deskriptor rubrik guru tidak tersedia.',
               'email_groups': len(email_first), 'connected_groups': len(roots),
               'questions': len({r['question'] for r in rows}),
               'duplicate_sets': sum(len(v) > 1 for v in duplicate_sets.values()),
               'cross_email_duplicate_sets': sum(len({rows[i]['email_group'] for i in v}) > 1 for v in duplicate_sets.values()),
               'source_sha256': source_hash,
               'input_features': ['soal','response_text','jawaban_referensi'],
               'grouping_note': 'Email dipakai sebagai kelompok pencegah kebocoran; tidak membuktikan identitas terverifikasi. Duplikat lintas email disatukan. Semua skenario memakai soal/referensi dan jawaban. Email hanya kelompok; nilai guru hanya target, bukan fitur encoder.'}
    return Bundle(rows, y, groups, source_hash, summary)


def run_folder(cfg, bundle):
    code_hash = NOTEBOOK_CODE_SHA256
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    key = hashlib.sha256(json.dumps({'source': bundle.source_sha256, 'config': semantic, 'code': code_hash}, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path(cfg['output']) / key
    folder.mkdir(parents=True, exist_ok=True)
    dump(folder / 'effective_config.json', cfg)
    return folder

In [ ]:
%%aes
# Bantuan membaca tabel: penamaan kolom, angka, dan penjelasan kata berulang.
LABELS = {'row_id':'ID Jawaban','word':'Kata','covered':'Tercakup','word_index':'Urutan Kata',
          'token_id':'ID Token','token':'Token','position':'Posisi Token','window':'Bagian Teks',
          'attention_mask':'Dibaca Model','skor_guru':'Nilai Guru','prediksi_mentah':'Prediksi Skor',
          'kategori_salinan':'Skor Dibulatkan','di_luar_rentang':'Di Luar Rentang',
          'Skenario':'Skenario','Fold':'Fold','AC2':'Gwet AC2','RMSE':'RMSE (poin)',
          'out_of_range_fraction':'Di Luar Rentang (%)','Status':'Keterangan',
          'teks_model':'Jawaban Siswa','teks_asli':'Teks Asli','score':'Nilai Guru',
          'role':'Penggunaan Data','connected_group':'Kelompok Jawaban',
          'input_format':'Format Input','response_text':'Jawaban Siswa','rater':'Nilai Guru','soal':'Soal',
          'sentence_index':'Urutan Kalimat','text':'Kalimat','fold':'Fold','scenario':'Skenario',
          'encoder':'Encoder','vector':'Panjang Vektor','head':'Lapisan Prediksi','output':'Jumlah Skor',
          'used_for':'Digunakan untuk','id':'Kode','title':'Judul',
          'rows':'Jumlah Jawaban','file':'File Hasil','stage':'Isi File','status':'Keadaan',
          'sheets_and_shapes':'Tabel yang Tersimpan','epoch':'Epoch',
          'train_mse_scaled':'Loss Latih (MSE)','target_loss':'Nilai untuk Menghitung Loss',
          'best_epoch_so_far':'Epoch Terbaik Sejauh Ini','epochs_without_improvement':'Epoch Tanpa Perbaikan',
          'selisih_mentah':'Selisih Prediksi − Guru','berubah':'Teks Dibersihkan?',
          'token_index':'Urutan Token','start':'Awal Teks','end':'Akhir Teks',
          'token_indices':'Urutan Potongan Token','token_ids':'ID Potongan Token',
          'covered_tokens':'Token yang Terbaca','total_tokens':'Jumlah Token',
          'window_coverage':'Jumlah Bagian yang Membaca Token',
          'occurrence_indices':'Posisi Kata yang Dibandingkan','same_token_ids':'ID Token Sama?',
          'initial_cosine':'Kemiripan Sebelum Konteks (Cosine)',
          'initial_max_abs_difference':'Selisih Terbesar Sebelum Konteks',
          'contextual_cosine':'Kemiripan Setelah Konteks (Cosine)',
          'contextual_max_abs_difference':'Selisih Terbesar Setelah Konteks',
          'contextual_allclose_1e-6':'Vektor Setelah Konteks Sama?',
          'undefined_reasons':'Alasan Metrik Tidak Tersedia','authors':'Penulis','year':'Tahun',
          'url':'Tautan','limitation':'Batas Penerapan','library':'Pustaka','version':'Versi',
          'name':'Pengaturan','value':'Nilai','field':'Keterangan','target':'Jenis Nilai',
          'Posisi Kata 1':'Posisi Kata A','Posisi Kata 2':'Posisi Kata B',
          'ID Token Sama':'ID Token Sama?','Vektor Awal Sama':'Vektor Sebelum Konteks Sama?',
          'Vektor Akhir Sama':'Vektor Setelah Konteks Sama?',
          'Kemiripan Vektor Akhir':'Kemiripan Setelah Konteks (Cosine)','Apa Artinya':'Penjelasan'}

REPEATED_COLUMNS = ['Model','Kata','Posisi Kata A','Posisi Kata B','ID Token Sama?',
                    'Vektor Sebelum Konteks Sama?','Vektor Setelah Konteks Sama?',
                    'Kemiripan Setelah Konteks (Cosine)','Penjelasan']

MODELS = {'S1':'RoBERTa','S2':'BGE-M3','S3':'Fusion RoBERTa + BGE-M3'}

ROLES = {'development':'Data latih (termasuk validasi silang)','test':'Data uji',
         'train':'Latih','validation':'Validasi'}

REPEATED_NOTES = [
    'Satu baris membandingkan kata yang sama pada dua posisi di jawaban contoh. Posisi dihitung mulai 1.',
    'Sebelum konteks = vektor dari tabel kosakata model. Setelah konteks = vektor setelah model membaca posisi dan kata di sekitarnya.',
    'Cosine mengukur kemiripan arah vektor: mendekati 1 berarti sangat mirip. Nilai yang terlihat 1 setelah pembulatan belum membuktikan seluruh angka vektornya sama.',
    'Kolom Vektor ... Sama? memeriksa selisih setiap komponen, dengan batas 0,000001. Ini batas perbandingan angka komputer, bukan tolerance skor AES.',
]

METRIC_NOTES = [
    'S1 = RoBERTa; S2 = BGE-M3; S3 = fusion keduanya. RMSE memakai satuan poin skor: lebih kecil berarti kesalahan lebih kecil.',
    'Pearson menunjukkan hubungan perubahan skor; QWK dan Gwet AC2 menunjukkan kesepakatan skor kategori. Korelasi tinggi saja belum menjamin prediksi mendekati nilai guru.',
    'Tanda — berarti hasil tidak tersedia, bukan nol. Baca keempat metrik bersama-sama.',
]

REPEATED_HEADERS=REPEATED_COLUMNS

LABELS.update({'input_segment':'Bagian Input','question_tokens':'Token Soal Awal',
    'question_kept':'Token Soal Dibaca','reference_tokens':'Token Referensi Awal',
    'reference_kept':'Token Referensi Dibaca','question_truncated':'Soal Terpotong?',
    'reference_truncated':'Referensi Terpotong?','context_tokens_kept':'Token Konteks Dibaca',
    'content_tokens':'Token Jawaban','capacity':'Kapasitas Jawaban per Window',
    'stride':'Langkah Window','starts':'Posisi Awal Window','windows':'Jumlah Window',
    'pair_special_tokens':'Token Khusus Pasangan','answer_tokens_uncovered':'Token Jawaban Tidak Tercakup',
    'context_max_tokens':'Batas Token Konteks Semua Encoder','question_max_tokens':'Batas Token Soal Semua Encoder'})

def friendly_header(name):
    match = re.fullmatch(r'v_(\d+)', str(name))
    return 'Komponen '+str(int(match[1])+1) if match else LABELS.get(name, str(name).replace('_',' '))


def friendly_value(name, value):
    if hasattr(value,'item'):
        try:value=value.item()
        except ValueError:pass
    if name == 'row_id' and isinstance(value,str):
        match = re.fullmatch(r'row_(\d+)',value)
        return f'J{int(match[1]):03d}' if match else value
    if name == 'role':
        return ROLES.get(value,value)
    if name == 'status':
        return {'ACTUAL EXECUTION':'Dihasilkan saat kode dijalankan',
                'DESIGN / CONFIGURATION':'Rancangan dan pengaturan model'}.get(value,value)
    if name == 'attention_mask' and value in (0,1):
        return 'Ya' if value else 'Tidak'
    if name == 'out_of_range_fraction' and isinstance(value,(int,float)):
        return value*100
    if name in ('word_index','sentence_index','token_index','position','window') and isinstance(value,(int,float)):
        return value+1
    if isinstance(value,bool):
        return 'Ya' if value else 'Tidak'
    return value


def repeated_word_rows(records,model):
    for row in records:
        positions=row.get('occurrence_indices')
        if isinstance(positions,str):positions=json.loads(positions)
        positions=positions or []
        same_ids=row.get('same_token_ids')
        initial=row.get('initial_max_abs_difference')
        same_initial=None if initial is None else abs(initial)<=1e-6
        same_context=row.get('contextual_allclose_1e-6')
        if same_ids is False:explanation='Kata sama, tetapi potongan token berbeda. Perhatikan tokenisasinya sebelum membandingkan vektor.'
        elif same_ids is None:explanation='Informasi kesamaan ID token belum tersedia.'
        elif same_context is None:explanation='Hasil perbandingan vektor akhir belum tersedia.'
        elif same_context:explanation='ID token sama; vektor akhir juga sama dalam toleransi numerik 0,000001.'
        else:explanation='ID token sama, tetapi vektor setelah membaca konteks berbeda. Posisi dan kata di sekitarnya dapat memengaruhi vektor.'
        label={'roberta':'RoBERTa','bge':'BGE-M3'}.get(model,model)
        yield [label,row.get('word'),positions[0]+1 if positions else None,positions[1]+1 if len(positions)>1 else None,
               friendly_value('',same_ids),friendly_value('',same_initial),friendly_value('',same_context),
               row.get('contextual_cosine'),explanation]

def preview_table(headers, records, title='', row_count=None, max_rows=8, max_columns=8):
    """Bentuk preview tanpa mengubah DataFrame/Excel sumber atau menghitung ulang model."""
    headers=list(headers)
    records=list(itertools.islice(records,max_rows))
    count=len(records) if row_count is None else row_count
    notes=[]
    original_columns=len(headers)
    if 'occurrence_indices' in headers:
        values=list(repeated_word_rows(records,''))
        # Nama model hanya ditampilkan bila memang diberikan oleh pemanggil.
        headers=REPEATED_HEADERS[1:]
        values=[r[1:] for r in values]
        notes=REPEATED_NOTES
        empty='Belum ada pasangan kata berulang yang bisa dibandingkan. Ini bukan error; lihat tabel vektor kata di atas.'
    elif {'file','stage','sheets_and_shapes'}.issubset(headers):
        # Dictionary ukuran tabel adalah rincian internal, bukan isi laporan belajar.
        headers=['File Hasil','Isi File','Keadaan']
        values=[[r.get('file'),r.get('stage'),friendly_value('status',r.get('status'))] for r in records]
        notes=['Daftar ini menunjukkan file yang sudah dibuat. Pilih lima laporan utama pada bagian unduhan untuk membaca hasilnya.']
        empty='Belum ada file hasil yang tersimpan.'
    else:
        selected=headers[:max_columns]
        values=[[friendly_value(key,row.get(key)) for key in selected] for row in records]
        headers=[friendly_header(key) for key in selected]
        empty='Belum ada baris hasil pada tahap ini. Tidak ada angka pengganti yang ditampilkan.'
        if 'Vektor Setelah Konteks Sama?' in headers or 'Vektor Akhir Sama' in headers:
            notes=REPEATED_NOTES
            empty='Belum ada pasangan kata berulang yang bisa dibandingkan. Ini bukan error; lihat tabel vektor kata di atas.'
        elif {'RMSE','Pearson','QWK','AC2'}.issubset(selected) or 'RMSE (poin)' in headers:
            notes=METRIC_NOTES
        elif 'prediksi_mentah' in selected or 'Prediksi S1' in headers:
            notes=['Prediksi Skor adalah nilai yang diberikan model. Nilai Guru adalah acuan jika tersedia.',
                   'Skor Dibulatkan adalah salinan untuk metrik kategori; prediksi asli tetap disimpan. Di Luar Rentang berarti prediksi melewati batas skor yang ditetapkan.']
        elif 'token_id' in selected or 'ID Token' in headers:
            notes=['Token adalah potongan teks yang dibaca model; satu kata bisa terpecah menjadi beberapa token. ID Token adalah nomor dalam kosakata model, bukan nilai siswa.',
                   'Posisi Token dan Bagian Teks dihitung mulai 1. Token khusus seperti <s> dan </s> menandai batas input.']
        components=sum(bool(re.fullmatch(r'v_\d+|Komponen \d+',str(h))) for h in selected)
        if components:
            notes=['Satu baris berisi satu vektor. Komponen 1, 2, dan seterusnya adalah angka penyusun vektor, bukan skor penilaian.',
                   'Tabel ini hanya memperlihatkan beberapa komponen pertama. Seluruh komponen tersimpan di Excel. Komponen yang sama nomornya pada RoBERTa dan BGE-M3 tidak memiliki arti yang otomatis sama.']
    return {'title':title,'columns':headers,'rows':values,'shape':[count,original_columns],
            'notes':notes,'empty_message':empty}


def cell_text(value):
    if value is None:return '—'
    if isinstance(value,bool):return 'Ya' if value else 'Tidak'
    if isinstance(value,float):
        return format(value,'.8g') if math.isfinite(value) else '—'
    if isinstance(value,dict):
        return '; '.join(f'{friendly_header(k)}: {cell_text(v)}' for k,v in value.items()) or '—'
    if isinstance(value,(list,tuple)):
        return ', '.join(cell_text(x) for x in value) or '—'
    return str(value)


def render_table(event):
    """HTML escapé dengan warna eksplisit agar terbaca pada Colab terang maupun gelap."""
    esc=lambda value:html.escape(cell_text(value))
    parts=['<section style="background:#f8fafc!important;color:#172035!important;padding:18px;border:1px solid #94a3b8;border-radius:12px;margin:12px 0">',
           '<h3 style="color:#172035!important;margin:0 0 10px">'+esc(event.get('title',''))+'</h3>']
    for note in event.get('notes',[]):parts.append('<p style="color:#334155!important;margin:6px 0">'+esc(note)+'</p>')
    if not event['rows']:
        parts.append('<p>'+esc(event.get('empty_message','Belum ada hasil.'))+'</p>')
    else:
        parts.append(f'<p style="color:#475569!important">Ditampilkan {len(event["rows"])} dari {event["shape"][0]} baris. Angka pada layar diringkas; nilai lengkap ada di Excel.</p>')
        th='background:#172035!important;color:#fff!important;padding:10px;border:1px solid #64748b;text-align:left'
        td='background:#f8fafc!important;color:#172035!important;padding:10px;border:1px solid #cbd5e1;vertical-align:top;min-width:90px;max-width:460px;white-space:normal;overflow-wrap:anywhere'
        parts.append('<div style="max-height:420px;overflow:auto"><table style="border-collapse:collapse"><thead><tr>')
        parts.extend('<th style="'+th+'">'+esc(x)+'</th>' for x in event['columns'])
        parts.append('</tr></thead><tbody>')
        for row in event['rows']:
            parts.append('<tr>'+''.join('<td style="'+td+'">'+esc(x)+'</td>' for x in row)+'</tr>')
        parts.append('</tbody></table></div>')
    parts.append('</section>')
    return ''.join(parts)

# Memeriksa nama GPU, status T4, dan VRAM; Colab berhenti sebelum training jika GPU belum aktif.
def check_gpu(require_gpu=False):
    """Periksa hardware yang benar-benar tersedia, tanpa memuat model pretrained."""
    available = torch.cuda.is_available()
    if not available:
        if require_gpu:
            raise RuntimeError('GPU belum aktif. Pilih Runtime → Change runtime type → T4 GPU, '
                               'lalu jalankan notebook dari atas. Training Colab tidak dimulai di CPU.')
        return {'name': 'CPU', 'cuda_available': False, 'is_t4': False, 'memory_gib': None}
    name = torch.cuda.get_device_name(0)
    return {'name': name, 'cuda_available': True, 'is_t4': 'T4' in name.upper(),
            'memory_gib': round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2)}


In [ ]:
%%aes
bundle = read_dataset(cfg)
output = run_folder(cfg, bundle)
show(pd.DataFrame([bundle.summary]), 'Audit dataset asli')
show(pd.DataFrame([dict(row_id=r['row_id'], soal=r['question'], referensi=r['reference'],
                       respons=r['text'], rater=r['score']) for r in bundle.rows]), 'Tiga input dan satu label')
print('Sumber dataset SHA256:',bundle.source_sha256,flush=True)

## 4 · Development/test dan fold kelompok

Resep split sama: GroupShuffleSplit seed2026 dengan128kandidat, pilih proporsi jumlah baris terdekat80:20 tanpa label; GroupKFold5 pada development. Untuk data300 yang sama, target240/60; jumlah aktual ditampilkan. Seluruh kelompok dipisahkan.

Test lama sudah dilihat dalam penelitian: evaluasi ini bersifat retrospektif. Keanggotaan dan teks harus dicocokkan sebelum dibandingkan dengan ZIP historis.

In [ ]:
%%aes
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest


def contexts_from_bundle(bundle):
    """Ambil soal/referensi tanpa membaca target skor guru."""
    contexts = []
    for row in bundle.rows:
        question, reference = row.get('question'), row.get('reference')
        if not isinstance(question, str) or not question.strip():
            raise ValueError('Context pair memerlukan soal yang tidak kosong.')
        if not isinstance(reference, str) or not reference.strip():
            raise ValueError('Context pair memerlukan jawaban referensi yang tidak kosong.')
        contexts.append((clean(question), clean(reference)))
    return contexts

In [ ]:
%%aes
manifest = make_split(bundle, cfg)
dump(output/'split_manifest.json', manifest)
show(pd.DataFrame([dict(development=len(manifest['development']), test=len(manifest['test']),
                       folds=len(manifest['folds']), development_groups=manifest['development_groups'],
                       test_groups=manifest['test_groups'])]), 'Partisi aktual')
print('Identitas split:',manifest['split_sha256'],flush=True)

## 5 · Tiga teks dan BGE-M3 frozen

Soal/referensi membentuk konteks maksimal192token (soal64), lalu window respons. RoBERTa maksimal512, BGE1024, overlap128. Semua token respons dicakup mengikuti helper E21. BGE memakai eval/no_grad/requires_grad=False; mean CLS antar-window lalu L2. BGE tidak masuk optimizer S3.

In [ ]:
%%aes
def mean_by_essay(cls, mapping, n):
    sums=cls.new_zeros((n,cls.shape[-1])).index_add(0,mapping,cls)
    counts=cls.new_zeros(n).index_add(0,mapping,cls.new_ones(len(mapping)))
    if (counts==0).any(): raise ValueError('Esai tanpa window')
    return sums / counts[:,None]


def context_token_ids(context, tokenizer, cfg):
    """Sisakan anggaran untuk soal dan referensi sekaligus."""
    if not isinstance(context, (tuple, list)) or len(context) != 2:
        raise ValueError('Konteks harus pasangan (soal, jawaban referensi).')
    question, reference = context
    if not all(isinstance(t, str) and t.strip() for t in context):
        raise ValueError('Isi soal dan jawaban referensi sebelum memprediksi.')
    limit, qlimit = cfg['context_max_tokens'], cfg['question_max_tokens']
    if not 1 <= qlimit < limit:
        raise ValueError('Anggaran token soal harus lebih kecil daripada konteks.')
    # Prefix menjadi bagian input; nilai guru tidak pernah dimasukkan.
    qids = tokenizer.encode('Soal: '+question, add_special_tokens=False)
    rids = tokenizer.encode('\nJawaban referensi: '+reference, add_special_tokens=False)
    question_kept = qids[:qlimit]
    reference_kept = rids[:limit-len(question_kept)]
    return question_kept+reference_kept, {
        'question_tokens': len(qids), 'question_kept': len(question_kept),
        'reference_tokens': len(rids), 'reference_kept': len(reference_kept),
        'question_truncated': len(question_kept) < len(qids),
        'reference_truncated': len(reference_kept) < len(rids),
        'context_tokens_kept': len(question_kept)+len(reference_kept),
    }


def pair_content_windows(text, context, tokenizer, maximum=512, overlap=128, cfg=None):
    """Informasi dan window sama; E22 menambah separator resmi pasangan."""
    if cfg is None or cfg.get('input_format') not in ('single_sequence', 'context_pair'):
        raise ValueError('Pilih format input E21 atau E22 sebelum tokenisasi.')
    context_ids, context_info = context_token_ids(context, tokenizer, cfg)
    ids = tokenizer.encode(text, add_special_tokens=False)
    answer_prefix = tokenizer.encode('\nJawaban siswa:\n', add_special_tokens=False)
    if not ids:
        raise ValueError('Jawaban tidak menghasilkan token.')
    paired = cfg['input_format'] == 'context_pair'
    specials = tokenizer.num_special_tokens_to_add(pair=paired)
    # Cadangkan overhead terbesar untuk KEDUA eksperimen. Starts/cakupan sama.
    reserved = max(tokenizer.num_special_tokens_to_add(pair=False),
                   tokenizer.num_special_tokens_to_add(pair=True))
    capacity = maximum-len(context_ids)-len(answer_prefix)-reserved
    if not 0 <= overlap < capacity:
        raise ValueError('Konteks dan penanda jawaban menyisakan kapasitas di bawah overlap.')
    starts = [0]
    while starts[-1]+capacity < len(ids):
        starts.append(starts[-1]+capacity-overlap)
    chunks, answer_positions, segments = [], [], []
    for start in starts:
        answer = ids[start:start+capacity]
        second = answer_prefix+answer
        if paired:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids, second)
            layout = tokenizer.build_inputs_with_special_tokens([-1]*len(context_ids),
                     [-3]*len(answer_prefix)+[-2]*len(answer))
        else:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids+second)
            layout = tokenizer.build_inputs_with_special_tokens(
                     [-1]*len(context_ids)+[-3]*len(answer_prefix)+[-2]*len(answer))
        positions = [i for i, value in enumerate(layout) if value == -2]
        if (len(prepared) != len(layout) or len(prepared) > maximum
                or len(prepared)-len(context_ids)-len(second) != specials
                or [prepared[i] for i in positions] != answer):
            raise ValueError('Template tokenizer tidak konsisten dengan format input.')
        chunks.append({'input_ids':prepared, 'attention_mask':[1]*len(prepared)})
        answer_positions.append(positions)
        segments.append(['Soal/referensi' if v == -1 else 'Jawaban siswa' if v == -2
                         else 'Penanda jawaban' if v == -3 else 'Token khusus' for v in layout])
    stats = {**context_info, 'input_format':cfg['input_format'], 'content_tokens':len(ids),
             'capacity':capacity, 'stride':capacity-overlap, 'starts':starts,
             'windows':len(chunks), 'pair_special_tokens':specials,
             'reserved_special_tokens':reserved, 'answer_prefix_tokens':len(answer_prefix),
             'answer_positions':answer_positions, 'segments':segments, 'answer_tokens_uncovered':0}
    return chunks, stats


def pair_batch_windows(texts, contexts, tokenizer, device='cpu', maximum=512, overlap=128, cfg=None):
    """Susun batch pasangan tanpa menukar konteks antarjawaban."""
    if len(texts) != len(contexts) or not texts:
        raise ValueError('Jumlah konteks harus sama dengan jumlah jawaban.')
    chunks, mapping, stats = [], [], []
    for index, (text, context) in enumerate(zip(texts, contexts)):
        values, detail = pair_content_windows(text, context, tokenizer, maximum, overlap, cfg)
        chunks.extend(values)
        mapping.extend([index]*len(values))
        stats.append(detail)
    batch = tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {key: value.to(device) for key, value in batch.items()}, torch.tensor(mapping, device=device), stats


def paired_pretrained_values(bundle, cfg, role):
    """Mean CLS antar-window; BGE dinormalisasi L2 setelah agregasi."""
    device = device_name(cfg)
    model, tok = encoder(cfg,role).to(device), tokenizer(cfg,role)
    model.eval()
    vectors, window_vectors, ids, coverage = [], [], [], []
    with torch.no_grad():
        for index, (text,context) in enumerate(zip(bundle.texts,contexts_from_bundle(bundle))):
            windows,mapping,details = pair_batch_windows([text],[context],tok,device,
                cfg[role+'_max_length'],cfg[role+'_overlap'],cfg)
            # Satu jawaban diproses bertahap agar jumlah window tidak meledakkan memori.
            cls_parts=[]
            for start in range(0,len(mapping),cfg['bge_batch']):
                part={k:v[start:start+cfg['bge_batch']] for k,v in windows.items()}
                cls_parts.append(model(**part).last_hidden_state[:,0,:].float())
            cls=torch.cat(cls_parts,dim=0)
            value=mean_by_essay(cls,mapping,1)
            if role=='bge':value=torch.nn.functional.normalize(value,dim=1)
            vectors.append(value.cpu().numpy())
            window_vectors.append(cls.cpu().numpy())
            row_id=bundle.rows[index].get('row_id',f'inference_{index}')
            ids.extend(f'{row_id}_window_{j}' for j in range(len(cls)))
            coverage.append({k:v for k,v in details[0].items() if k not in ('answer_positions','segments')})
            if (index+1)%25==0 or index+1==len(bundle.rows):
                print(f'{role}: tiga input {index+1}/{len(bundle.rows)} jawaban selesai.',flush=True)
    values=np.concatenate(vectors).astype(np.float32)
    if role=='bge' and any(p.requires_grad for p in model.parameters()):
        raise ValueError('Encoder BGE harus frozen.')
    del model,tok
    release()
    return values,np.concatenate(window_vectors),ids,coverage

In [ ]:
%%aes
def device_name(cfg):
    requested = cfg['device']
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA diminta tetapi tidak tersedia; pilih GPU di Colab atau device=cpu.')
    return ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested


def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class ExpectedPoolerNotice(logging.Filter):
    """Hanya saring pemberitahuan dua bobot pooler yang memang tidak dipakai."""
    def filter(self, record):
        import ast
        import re
        message = record.getMessage()
        if not message.startswith('Some weights of the model checkpoint at ') or 'were not used when initializing' not in message:
            return True
        match = re.search(r"\['[^\n]+?\]", message)
        if match is None:
            return True
        try:
            keys = ast.literal_eval(match.group())
        except (ValueError, SyntaxError):
            return True
        return set(keys) != {'pooler.dense.bias', 'pooler.dense.weight'}


def encoder(cfg, role):
    # Ambil nama checkpoint dan revision model yang dipilih: roberta atau bge.
    spec = cfg[role]
    logger = logging.getLogger('transformers.modeling_utils')
    notice = ExpectedPoolerNotice()
    logger.addFilter(notice)
    try:
        # Muat arsitektur dan bobot pretrained. Ini baru inisialisasi encoder, belum latihan memakai nilai guru.
        result = AutoModel.from_pretrained(spec['checkpoint'], revision=spec['revision'], add_pooling_layer=False)
    finally:
        logger.removeFilter(notice)
    # Periksa panjang vektor: RoBERTa 768 angka, BGE-M3 1024 angka.
    expected = 768 if role == 'roberta' else 1024
    if result.config.hidden_size != expected:
        raise ValueError(f'{role}: dimensi checkpoint tidak cocok dengan {expected}.')
    # Parameter RoBERTa ikut diperbarui; parameter BGE tidak ikut diperbarui.
    # RoBERTa boleh belajar; bobot encoder BGE tidak diperbarui.
    result.requires_grad_(role == 'roberta')
    return result


def tokenizer(cfg, role, fast=False):
    # Muat tokenizer dari checkpoint yang sama agar ID token cocok dengan kosakata model.
    result = AutoTokenizer.from_pretrained(cfg[role]['checkpoint'], revision=cfg[role]['revision'],
                                           use_fast=fast if role == 'roberta' else True)
    if role == 'roberta' and not fast and result.num_special_tokens_to_add(False) != 2:
        raise ValueError('RoBERTa slow tokenizer harus menambahkan dua special token.')
    return result

In [ ]:
%%aes
def bge_features(bundle, cfg, output):
    """Cache berubah jika soal, referensi, jawaban, atau resep token berubah."""
    contexts=contexts_from_bundle(bundle)
    contract={'model':cfg['bge'], 'texts':bundle.texts, 'contexts':contexts,
              'maximum':cfg['bge_max_length'],'overlap':cfg['bge_overlap'],
              'context_max_tokens':cfg['context_max_tokens'],'question_max_tokens':cfg['question_max_tokens'],
              'policy':'three_inputs_mean_CLS_then_L2_v2', 'input_format':cfg['input_format']}
    key=hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:20]
    cache=Path(cfg['output'])/'cache'/f'bge_three_inputs_{key}.npz'
    metadata=cache.with_suffix('.json')
    if cache.exists() and metadata.exists():
        proof=json.loads(metadata.read_text())
        if proof['file_sha256']!=hashlib.sha256(cache.read_bytes()).hexdigest():
            raise ValueError('Cache BGE berubah; tidak dipakai.')
        with np.load(cache,allow_pickle=False) as data:values=data['features']
        reused=True
    else:
        values,_,_,coverage=paired_pretrained_values(bundle,cfg,'bge')
        cache.parent.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(cache,features=values)
        proof={'model':cfg['bge'],'shape':list(values.shape),'frozen':True,
               'pooling':'mean paired-window CLS then L2','max_length':cfg['bge_max_length'],
               'feature_source':'soal + jawaban_referensi + response_text; no teacher score', 'input_format':cfg['input_format'],
               'answer_tokens_uncovered':sum(s['answer_tokens_uncovered'] for s in coverage),
               'context_truncated_rows':sum(s['question_truncated'] or s['reference_truncated'] for s in coverage),
               'file_sha256':hashlib.sha256(cache.read_bytes()).hexdigest()}
        dump(metadata,proof)
        reused=False
    if values.shape!=(len(bundle.rows),1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4):
        raise ValueError('Fitur BGE shape/normalisasi tidak sah.')
    dump(Path(output)/'bge_encoding.json',{**proof,'reused':reused})
    return torch.from_numpy(values)
bge = bge_features(bundle, cfg, output)
X = bge.numpy().astype(np.float32)
np.save(output/'bge_frozen.npy', X)
show(pd.DataFrame([dict(shape=str(X.shape), norm_min=float(np.linalg.norm(X,axis=1).min()),
    norm_max=float(np.linalg.norm(X,axis=1).max()), encoder_trained=False)]), 'BGE frozen bersama untuk S3 dan baseline S2')
print('BGE selesai; model encoder sudah dilepas dari memori. Selanjutnya fine-tuning representasi S3 per split.',flush=True)

## 6 · Fusi E21 dan training hanya pada train

MLP bantu1024→256→1 mempelajari representasi bersama RoBERTa, LayerNorm dan proyeksi BGE. Setelah25epoch tetap, vektor concat1024 dipakai untuk fit Ridge. Ridge tidak melakukan backpropagation ke encoder.

Seluruh target di luar train ditutupi NaN saat fungsi training dipanggil. Tidak ada early stopping memakai outer validation atau test. Checkpoint epoch menyimpan optimizer/RNG agar dapat dilanjutkan selama disk runtime masih ada.

In [ ]:
%%aes
class SingleScoreModel(nn.Module):
    def __init__(self, scenario, roberta=None, dropout=.1):
        super().__init__()
        if scenario not in ('S1', 'S2', 'S3'):
            raise ValueError('Skenario harus S1, S2, atau S3')
        if scenario != 'S2' and roberta is None:
            raise ValueError('RoBERTa diperlukan oleh S1/S3.')
        # Simpan skenario dan encoder RoBERTa. S2 memakai vektor BGE yang sudah dihitung, jadi tidak memuat RoBERTa.
        self.scenario, self.roberta = scenario, roberta
        if roberta is not None:
            # Fine-tuning: bobot RoBERTa boleh berubah ketika optimizer menjalankan langkah latihan.
            roberta.requires_grad_(True)
        self.dropout = nn.Dropout(dropout)
        if scenario == 'S1':
            # S1: 768 koordinat RoBERTa dipetakan ke satu prediksi kontinu.
            self.head = nn.Linear(768, 1)
        elif scenario == 'S2':
            # MLP mempelajari pemetaan vektor menjadi skor; GELU memungkinkan hubungan nonlinier.
            self.head = nn.Sequential(nn.Linear(1024, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1))
        else:
            # LayerNorm tambahan pada vektor jawaban RoBERTa, khusus cabang fusion.
            self.roberta_norm = nn.LayerNorm(768)
            # S3: padatkan 1024 angka BGE menjadi 256 angka sebelum digabung.
            self.projection = nn.Sequential(nn.Linear(1024, 256), nn.LayerNorm(256))
            # MLP S3: linear 1024 ke 256, GELU, dropout, lalu satu skor.
            self.head = nn.Sequential(nn.Linear(1024, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1))

    def forward(self, windows=None, mapping=None, n_essays=None, bge=None, return_features=False):
        features = {}
        if self.scenario != 'S2':
            # Baca token pada setiap window, lalu ambil vektor token pertama (CLS) sebagai ringkasannya.
            cls = self.roberta(**windows).last_hidden_state[:, 0, :]
            # Rata-ratakan vektor CLS semua window milik jawaban yang sama.
            essay = mean_by_essay(cls, mapping, n_essays)
            features.update(roberta=essay, roberta_window_cls=cls)
        if self.scenario == 'S1':
            # Lapisan prediksi memetakan vektor menjadi satu angka; saat latihan angka ini masih memakai skala target 0–1.
            prediction = self.head(self.dropout(essay))
        elif self.scenario == 'S2':
            if bge is None or bge.ndim != 2 or bge.shape[1] != 1024:
                raise ValueError('BGE memerlukan [N,1024]')
            features['bge'] = bge
            # Lapisan prediksi memetakan vektor menjadi satu angka; saat latihan angka ini masih memakai skala target 0–1.
            prediction = self.head(bge)
        else:
            if bge is None or bge.shape != (n_essays, 1024):
                raise ValueError('BGE memerlukan [N,1024]')
            features['bge'] = bge
            # Simpan vektor asli dan hasil normalisasi agar setiap tahap bisa diperiksa.
            # Normalisasi tambahan pada 768 komponen RoBERTa sebelum fusion; vektor asli tetap disimpan.
            features['roberta_layernorm'] = self.roberta_norm(essay)
            # Proyeksi BGE ikut belajar, walaupun bobot encoder BGE tetap frozen.
            features['projection'] = self.projection(bge)
            # Sambung 768 angka RoBERTa setelah LayerNorm dan 256 angka proyeksi BGE menjadi vektor fusion 1024 angka.
            features['concat'] = torch.cat([features['roberta_layernorm'], features['projection']], dim=1)
            # Lapisan prediksi memetakan vektor menjadi satu angka; saat latihan angka ini masih memakai skala target 0–1.
            # Dropout sudah berada di hidden MLP; concat masuk langsung ke head.
            prediction = self.head(features['concat'])
        if prediction.ndim != 2 or prediction.shape[1] != 1:
            raise ValueError('Dataset satu skor membutuhkan output [N,1].')
        return (prediction, features) if return_features else prediction


class Engine:
    def __init__(self, model, texts, bge, cfg, device, roberta_tokenizer=None, contexts=None):
        self.model, self.texts, self.bge, self.cfg, self.device = model, texts, bge, cfg, device
        self.contexts = contexts
        if contexts is None or len(contexts)!=len(texts):
            raise ValueError('Semua skenario memerlukan satu pasangan soal/referensi per jawaban.')
        self.tokenizer = roberta_tokenizer or (tokenizer(cfg, 'roberta') if model.scenario != 'S2' else None)

    def forward(self, indices, return_features=False):
        ids = [int(i) for i in indices]
        features = self.bge[ids].to(self.device) if self.model.scenario != 'S1' else None
        if self.model.scenario == 'S2':
            return self.model(bge=features, return_features=return_features)
        # Pecah jawaban panjang menjadi window; mapping menghubungkan setiap window dengan jawaban asalnya.
        # S1 dan S3 membaca tiga teks yang sama; S2 memakai fitur BGE tiga input.
        windows,mapping,_=pair_batch_windows([self.texts[i] for i in ids],[self.contexts[i] for i in ids],
            self.tokenizer,self.device,self.cfg['roberta_max_length'],self.cfg['roberta_overlap'],self.cfg)
        return self.model(windows=windows, mapping=mapping, n_essays=len(ids), bge=features,
                          return_features=return_features)

    def predict(self, indices):
        # Mode prediksi: nonaktifkan dropout. Pembacaan di bawah memakai no_grad.
        self.model.eval()
        values = []
        # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
        with torch.no_grad():
            for start in range(0, len(indices), self.cfg['micro_batch']):
                ids = indices[start:start+self.cfg['micro_batch']]
                values.append(self.forward(ids).reshape(-1).float().cpu().numpy())
        scaled = np.concatenate(values)
        # Kembalikan skala asli sebelum RMSE/Pearson maupun tampilan prediksi.
        # Kembalikan prediksi ke skala nilai guru sebelum ditampilkan atau dihitung RMSE/Pearson.
        return scaled*(self.cfg['score_max']-self.cfg['score_min'])+self.cfg['score_min']

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_model(scenario, cfg):
    # S1/S3 memuat RoBERTa; S2 hanya membuat lapisan prediksi di atas vektor BGE frozen.
    return SingleScoreModel(scenario, encoder(cfg, 'roberta') if scenario != 'S2' else None,
                            dropout=cfg['dropout'])

In [ ]:
%%aes
def gwet_ac2(teacher, predicted, categories):
    a=np.asarray(teacher);b=np.asarray(predicted);c=np.asarray(categories,dtype=float)
    if len(a)!=len(b) or len(a)<2: raise ValueError('AC2 perlu minimal dua pasangan')
    if not np.isin(a,c).all() or not np.isin(b,c).all(): raise ValueError('Kategori AC2 tidak sah')
    q=len(c); weights=1-((c[:,None]-c[None,:])/(c.max()-c.min()))**2
    ai=np.searchsorted(c,a); bi=np.searchsorted(c,b)
    pa=weights[ai,bi].mean()
    pi=(np.bincount(ai,minlength=q)+np.bincount(bi,minlength=q))/(2*len(a))
    pe=weights.sum()*np.sum(pi*(1-pi))/(q*(q-1))
    return float((pa-pe)/(1-pe))

def metrics(teacher, prediction, cfg):
    y, p = np.asarray(teacher, dtype=float), np.asarray(prediction, dtype=float)
    if y.ndim != 1 or y.shape != p.shape or len(y) == 0:
        raise ValueError('Metrik satu skor memerlukan dua vektor [N] yang sama dan tidak kosong.')
    if not np.isfinite(y).all() or not np.isfinite(p).all():
        raise ValueError('Label/prediksi tidak finite.')
    cats = np.arange(cfg['score_min'], cfg['score_max']+1)
    if not np.isin(y, cats).all():
        raise ValueError('Skor guru harus kategori integer pada rentang yang dinyatakan.')
    # Buat SALINAN skor sah untuk agreement; prediksi mentah tetap utuh.
    clipped = np.clip(p, cfg['score_min'], cfg['score_max'])
    # Tie .5 memilih kategori lebih rendah, sama dengan kontrak konversi proyek.
    ordinal = cats[np.abs(clipped[:, None]-cats[None, :]).argmin(axis=1)]
    result = {'RMSE': float(np.sqrt(np.mean((y-p)**2))), 'Pearson': None,
              'QWK': None, 'AC2': None, 'undefined_reasons': {},
              'out_of_range_fraction': float(np.mean((p < cats[0]) | (p > cats[-1])))}
    if len(y) > 1 and np.ptp(y) > 0 and np.ptp(p) > 0:
        result['Pearson'] = float(pearsonr(y, p).statistic)
    else:
        result['undefined_reasons']['Pearson'] = 'Perlu minimal dua pasangan dan kedua vektor tidak konstan.'
    if len(y) > 1:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            # QWK memakai jarak kategori kuadratik, bukan vektor embedding.
            qwk = cohen_kappa_score(y, ordinal, labels=cats, weights='quadratic')
        if np.isfinite(qwk):
            result['QWK'] = float(qwk)
        else:
            result['undefined_reasons']['QWK'] = 'Expected disagreement nol; undefined, bukan nol.'
        # AC2 memakai kategori yang sama dengan QWK dan koreksi peluang berbeda.
        result['AC2'] = gwet_ac2(y, ordinal, cats)
    else:
        result['undefined_reasons'].update(QWK='Perlu minimal dua pasangan.', AC2='Perlu minimal dua pasangan.')
    return result

In [ ]:
%%aes
def optimizer_groups(model, cfg):
    # Tiap parameter hanya masuk satu kelompok; encoder BGE frozen berada di luar model ini.
    buckets = {'roberta': [], 'projection': [], 'head': []}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        group = 'roberta' if name.startswith('roberta.') else 'projection' if name.startswith('projection.') else 'head'
        buckets[group].append(parameter)
    rates = {'roberta': cfg['s3_encoder_lr'] if model.scenario=='S3' else cfg['encoder_lr'], 'projection': cfg['projection_lr'], 'head': cfg['head_lr']}
    # LayerNorm tambahan RoBERTa baru dibuat, sehingga ikut kelompok head yang baru belajar.
    return [{'name': name, 'params': params, 'lr': rates[name]}
            for name, params in buckets.items() if params]


def save_training_progress(path,payload):
    """Dua slot pada disk runtime lokal; tidak membuat satu file per epoch."""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    previous=json.loads(path.read_text())['snapshots'] if path.exists() else []
    slot='A' if payload['epoch']%2 else 'B'
    target=path.with_name(f'{path.stem}_slot_{slot}.pt')
    # Tulis hanya slot yang lebih lama. Bila terputus, slot terbaru lain masih utuh.
    torch.save(payload,target)
    checksum=digest(target)
    snapshots=[{'file':target.name,'sha256':checksum}]+[r for r in previous if r['file']!=target.name][:1]
    dump(path,{'scope':SCOPE,'snapshots':snapshots})


def load_training_progress(path, contract):
    path = Path(path)
    if not path.exists():
        return None
    manifest = json.loads(path.read_text())
    if manifest.get('scope') != SCOPE:
        raise ValueError('Checkpoint progres bukan workflow AES ini.')
    for snapshot in manifest['snapshots']:
        source = path.parent/snapshot['file']
        if source.name != snapshot['file']:
            raise ValueError('Nama file checkpoint progres tidak sah.')
        if not source.exists() or digest(source) != snapshot['sha256']:
            print('Checkpoint terbaru tidak lengkap; memeriksa cadangan epoch sebelumnya.', flush=True)
            continue
        payload = torch.load(source, map_location='cpu', weights_only=True)
        if payload.get('contract') != contract:
            raise ValueError('Data, fold, konfigurasi atau perangkat checkpoint berbeda. Progres tidak dipakai untuk percobaan lain.')
        return payload
    raise RuntimeError('Tidak ada checkpoint progres yang utuh. Hasil lama dipertahankan; training tidak diulang diam-diam.')


def fit(scenario, training, validation, bundle, bge, cfg, epochs, seed, checkpoint=None):
    seed_everything(seed)
    device = device_name(cfg)
    # Buat model baru untuk fold ini, lalu tempatkan di GPU atau CPU yang tersedia.
    model = make_model(scenario, cfg).to(device)
    engine = Engine(model, bundle.texts, bge, cfg, device, contexts=contexts_from_bundle(bundle))
    groups = optimizer_groups(model, cfg)
    learning_rates = {group['name']: group['lr'] for group in groups}
    # AdamW memperbarui kelompok parameter sesuai learning rate masing-masing.
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader([int(i) for i in training], batch_size=cfg['effective_batch'],
                        shuffle=True, generator=generator, num_workers=0)
    total_steps = len(loader)*epochs
    warmup = min(int(cfg['warmup_ratio']*total_steps), max(0, total_steps-1))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)
    use_cuda = device == 'cuda'
    dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda and dtype == torch.float16)
    selected_parameters = {}
    for name, parameter in model.named_parameters():
        group = name.split('.')[0]
        if parameter.requires_grad and group not in selected_parameters:
            selected_parameters[group] = (name, parameter.detach().reshape(-1)[:32].cpu().clone())
    best_state, best_epoch, best_rmse = None, 0, float('inf')
    patience = cfg['early_stopping_patience']
    without_improvement, early_stopped = 0, False
    history, active_steps, optimizer_steps, skipped_steps = [], 0, 0, 0
    start_epoch = 1
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    contract = {'format': 'epoch_resume_v1', 'source': bundle.source_sha256, 'scenario': scenario,
                'training': [int(i) for i in training], 'validation': None if validation is None else [int(i) for i in validation],
                'config': semantic, 'epochs': epochs, 'seed': seed,
                'torch': str(torch.__version__), 'device': device,
                'precision': str(dtype) if use_cuda else 'float32',
                'gpu': torch.cuda.get_device_name() if use_cuda else None}
    saved = load_training_progress(checkpoint, contract) if checkpoint is not None else None
    if saved is not None:
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        generator.set_state(saved['loader_generator'])
        selected_parameters = saved['selected_parameters']
        best_state, best_epoch, best_rmse = saved['best_state'], saved['best_epoch'], saved['best_rmse']
        history, without_improvement = saved['history'], saved['without_improvement']
        active_steps, optimizer_steps, skipped_steps = saved['active_steps'], saved['optimizer_steps'], saved['skipped_steps']
        early_stopped = saved['early_stopped']
        start_epoch = epochs+1 if early_stopped else saved['epoch']+1
        random.setstate(saved['python_rng'])
        rng = saved['numpy_rng']
        np.random.set_state((rng[0], np.asarray(rng[1], dtype=np.uint32), rng[2], rng[3], rng[4]))
        torch.set_rng_state(saved['torch_rng'])
        if use_cuda:
            torch.cuda.set_rng_state_all(saved['cuda_rng'])
        print(f'{scenario}: progres epoch {saved["epoch"]}/{epochs} ditemukan; '
              + (f'lanjut epoch {start_epoch}.' if start_epoch <= epochs else 'latihan sudah selesai, memulihkan hasilnya.'), flush=True)
        del saved
    for epoch in range(start_epoch, epochs+1):
        model.train()
        loss_total = 0.
        for effective in loader:
            batch = effective.tolist()
            optimizer.zero_grad(set_to_none=True)
            for start in range(0, len(batch), cfg['micro_batch']):
                ids = batch[start:start+cfg['micro_batch']]
                # Ubah nilai guru dari skala 0–100 menjadi 0–1 hanya untuk menghitung loss.
                targets = torch.as_tensor((bundle.y[ids]-cfg['score_min'])/(cfg['score_max']-cfg['score_min']),
                                          dtype=torch.float32, device=device).reshape(-1, 1)
                with torch.autocast(device_type=device, dtype=dtype, enabled=use_cuda):
                    prediction = engine.forward(ids).float()
                    if prediction.shape != targets.shape:
                        raise ValueError('Shape prediksi/target harus [B,1], tanpa broadcasting.')
                    # MSE mengukur selisih kuadrat antara prediksi dan target; optimizer berusaha memperkecilnya.
                    loss = torch.mean((prediction-targets)**2)
                if not torch.isfinite(loss):
                    raise RuntimeError('Loss nonfinite; run dihentikan, tidak diklaim berhasil.')
                # Hitung gradien dan bobot microbatch sesuai ukuran batch efektif aktual.
                scaler.scale(loss*(len(ids)/len(batch))).backward()
                loss_total += loss.item()*len(ids)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
            lr_active = any(g['lr'] > 0 for g in optimizer.param_groups)
            previous_scale = scaler.get_scale()
            # Perbarui bobot; GradScaler bisa melewati langkah yang overflow pada AMP.
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.is_enabled() and scaler.get_scale() < previous_scale
            if skipped:
                skipped_steps += 1
            else:
                active_steps += int(lr_active)
                scheduler.step()
                optimizer_steps += 1
        row = {'epoch': epoch, 'train_mse_scaled': loss_total/len(training),
               'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
               'effective_batch_target': cfg['effective_batch'], 'micro_batch': cfg['micro_batch']}
        if validation is not None:
            predicted = engine.predict(validation)
            row['validation'] = metrics(bundle.y[validation], predicted, cfg)
            # Simpan epoch dengan RMSE validasi terkecil; data uji akhir tidak ikut memilih epoch.
            if row['validation']['RMSE'] < best_rmse:
                best_rmse, best_epoch = row['validation']['RMSE'], epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                without_improvement = 0
            else:
                without_improvement += 1
            row.update(best_epoch_so_far=best_epoch, epochs_without_improvement=without_improvement)
            message = (f" | RMSE validasi: {row['validation']['RMSE']:.4f} poin"
                       f" | terbaik: epoch {best_epoch}, {best_rmse:.4f} poin"
                       f" | tanpa perbaikan: {without_improvement}/{patience}")
        else:
            best_epoch = epoch
            message = ' | pelatihan final; data uji belum dibuka'
        history.append(row)
        print(f'{scenario} · epoch {epoch}/{epochs} · loss latih (MSE, target 0–1): {row["train_mse_scaled"]:.6f}'+message, flush=True)
        # Hanya fold validasi yang boleh menghentikan latihan; data uji tidak dibaca.
        if validation is not None and without_improvement >= patience and epoch < epochs:
            early_stopped = True
        if checkpoint is not None:
            rng = np.random.get_state()
            # Progres menyimpan keadaan belajar, bukan bobot terbaik saja.
            save_training_progress(checkpoint, {
                'contract': contract, 'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                'loader_generator': generator.get_state(), 'python_rng': random.getstate(),
                'numpy_rng': (rng[0], rng[1].tolist(), rng[2], rng[3], rng[4]),
                'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all() if use_cuda else [],
                'selected_parameters': selected_parameters, 'best_state': best_state,
                'best_epoch': best_epoch, 'best_rmse': best_rmse, 'history': history,
                'without_improvement': without_improvement, 'early_stopped': early_stopped,
                'active_steps': active_steps, 'optimizer_steps': optimizer_steps, 'skipped_steps': skipped_steps})
            print(f'{scenario}: progres epoch {epoch} tersimpan di disk lokal.', flush=True)
        if early_stopped:
            print(f'{scenario}: early stopping setelah {epoch} epoch; RMSE validasi tidak membaik selama {patience} epoch. '
                  f'Bobot epoch {best_epoch} dipulihkan.', flush=True)
            break
    if active_steps == 0:
        raise RuntimeError('Tidak ada pembaruan dengan LR aktif.')
    named = dict(model.named_parameters())
    changed = {group: not torch.equal(before, named[name].detach().reshape(-1)[:32].cpu())
               for group, (name, before) in selected_parameters.items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    info = {'scenario': scenario, 'best_epoch': best_epoch, 'epochs_completed': len(history),
            'max_epochs': epochs, 'early_stopping_patience': patience if validation is not None else None,
            'early_stopped': early_stopped, 'optimizer_learning_rates': learning_rates,
            'best_validation_rmse': best_rmse if validation is not None else None,
            'stop_reason': ('RMSE validasi tidak membaik selama batas kesabaran' if early_stopped else
                            'Batas maksimum epoch tercapai' if validation is not None else
                            'Jumlah epoch tetap yang ditentukan sebelum run selesai'),
            'history': history, 'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
            'parameter_samples_changed': changed, 'BGE_encoder_trained': False,
            'selection': 'minimum validation RMSE in original score units' if validation is not None else 'fixed epoch budget declared before run'}
    return model, engine, info

## 7 · Nested CV untuk encoder supervised dan Ridge

Untuk setiap outer fold, inner validation juga tidak boleh pernah menjadi label training representasi. Karena itu encoder/proyeksi dilatih pada3blok fold, Ridge dipasang di3blok tersebut, lalu alpha dibandingkan pada blok inner validation. Pasangan dua fold yang dikeluarkan menghasilkan train yang sama dalam dua arah; cache dipakai bersama sehingga10fit cukup untuk20validasi inner.

5fit outer memberi prediksi OOF yang tidak dilatih pada kelompoknya. Alpha final dipilih melalui5fold development menggunakan representasi outer masing-masing. Model final baru dilatih pada seluruh development, lalu test dievaluasi.

Cache berisi fitur dan identitas training. Dua snapshot optimizer sementara milik tahap ini dihapus hanya setelah cache selesai dan checksum tersimpan; tahap terputus mempertahankan snapshot resume. Checkpoint encoder final tetap disimpan. Folder eksperimen lama tidak disentuh.

In [ ]:
%%aes
def write_readout_workbook(path, sheets):
    # Strings are literal text, never spreadsheet formulas from uploaded essays.
    book = Workbook(write_only=True)
    for title, data in sheets.items():
        frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
        sheet = book.create_sheet(title[:31])
        sheet.append([str(v) for v in frame.columns])
        for record in frame.itertuples(index=False, name=None):
            cells = []
            for value in record:
                if isinstance(value, (list, dict, tuple)):
                    value = json.dumps(value, ensure_ascii=False)
                if isinstance(value, np.generic):
                    value = value.item()
                if isinstance(value, float) and not math.isfinite(value):
                    value = None
                cell = WriteOnlyCell(sheet, value=value)
                if isinstance(value, str):
                    cell.data_type = 's'
                cells.append(cell)
            sheet.append(cells)
    book.save(path)

In [ ]:
%%aes
def validate_ridge_inputs(bundle, X, manifest, cfg):
    X = np.asarray(X, dtype=np.float64)
    if X.shape != (len(bundle.rows), 1024) or not np.isfinite(X).all():
        raise ValueError('Ridge perlu fitur BGE finite [jumlah jawaban, 1024].')
    if not np.allclose(np.linalg.norm(X, axis=1), 1., atol=1e-4):
        raise ValueError('Fitur harus menggunakan L2 per jawaban yang sama dengan acuan.')
    alphas = cfg['ridge_alphas']
    if not alphas or len(set(alphas)) != len(alphas) or any(
            isinstance(a, bool) or not math.isfinite(a) or a <= 0 for a in alphas):
        raise ValueError('Alpha harus unik, positif, finite dan ditentukan sebelum evaluasi.')
    dev, test = set(manifest['development']), set(manifest['test'])
    if not dev or not test or dev & test or dev | test != set(range(len(X))):
        raise ValueError('Development dan test harus membagi seluruh baris tanpa overlap.')
    if len(dev) != len(manifest['development']) or len(test) != len(manifest['test']):
        raise ValueError('Indeks partisi duplikat.')
    if set(bundle.groups[list(dev)]) & set(bundle.groups[list(test)]):
        raise ValueError('Kelompok menyeberang development dan test.')
    covered = []
    for fold in manifest['folds']:
        tr, va = set(fold['train']), set(fold['validation'])
        if not tr or not va or tr & va or tr | va != dev:
            raise ValueError('Fold harus membagi development; test tidak boleh masuk fold.')
        if len(tr) != len(fold['train']) or len(va) != len(fold['validation']):
            raise ValueError('Indeks fold duplikat.')
        if set(bundle.groups[list(tr)]) & set(bundle.groups[list(va)]):
            raise ValueError('Kelompok menyeberang train dan validation.')
        covered.extend(fold['validation'])
    if len(manifest['folds']) < 3 or len(covered) != len(dev) or set(covered) != dev:
        raise ValueError('Setiap development wajib menjadi validation tepat sekali; minimal3fold.')
    return X


def ridge_metric_row(partition, teacher, prediction, cfg, **extra):
    m = metrics(teacher, prediction, cfg)
    errors = np.asarray(prediction) - np.asarray(teacher)
    absolute = np.abs(errors)
    m.update(n=len(errors), MAE=float(absolute.mean()), bias=float(errors.mean()),
             median_absolute_error=float(np.median(absolute)), p90_absolute_error=float(np.quantile(absolute, .90)),
             p95_absolute_error=float(np.quantile(absolute, .95)), max_absolute_error=float(absolute.max()))
    for threshold in (1, 2, 5, 10):
        m['within' + str(threshold)] = int(np.sum(absolute <= threshold))
    reasons = json.dumps(m.pop('undefined_reasons'), ensure_ascii=False)
    return dict(partition=partition, **extra, **m, undefined_reasons=reasons)

In [ ]:
%%aes
"""Standalone S3 feature fusion with fold-safe supervised representations and Ridge.

The builder embeds this runtime together with pinned E21 helpers. No research data.
"""
from sklearn.linear_model import Ridge

S3_MODELS = ('S3_RIDGE', 'S3_MLP_CONTROL', 'S2_RIDGE_FIXED')
S3_BOOKS = ('01_Data_dan_Split.xlsx', '02_Fitur_Fusion_dan_Model.xlsx',
            '03_Prediksi_OOF_dan_Test.xlsx', '04_Evaluasi_dan_Alpha.xlsx',
            '05_Riwayat_dan_Referensi.xlsx')
S3_REFERENCES = [
    dict(component='Ridge', authors='Hoerl & Kennard (1970)',
         title='Ridge Regression: Biased Estimation for Nonorthogonal Problems',
         url='https://doi.org/10.1080/00401706.1970.10488634',
         limit='Dasar penalti L2, bukan bukti S3 Ridge pasti mengungguli MLP atau alpha tertentu optimal.'),
    dict(component='Pemisahan seleksi dan evaluasi', authors='Cawley & Talbot (2010)',
         title='On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation',
         url='https://www.jmlr.org/papers/v11/cawley10a.html',
         limit='Mendasari nested CV; tidak menghapus bias penelitian dari test historis yang sudah dilihat.'),
    dict(component='Pembaca representasi tetap', authors='Conneau & Kiela (2018)',
         title='SentEval: An Evaluation Toolkit for Universal Sentence Representations',
         url='https://aclanthology.org/L18-1269/',
         limit='Prinsip evaluasi embedding downstream; bukan resep khusus BGE-M3 Ridge AES.'),
    dict(component='BGE-M3', authors='Chen et al. (2024)', title='BGE M3-Embedding',
         url='https://arxiv.org/abs/2402.03216', limit='Dasar encoder frozen, bukan head atau parameter optimum AES.'),
    dict(component='RoBERTa', authors='Liu et al. (2019)', title='RoBERTa: A Robustly Optimized BERT Pretraining Approach',
         url='https://arxiv.org/abs/1907.11692', limit='Arsitektur/pretraining; checkpoint Indonesia dan LR diwarisi source E21.'),
    dict(component='AdamW', authors='Loshchilov & Hutter (2019)', title='Decoupled Weight Decay Regularization',
         url='https://arxiv.org/abs/1711.05101', limit='Dasar optimizer representasi, bukan penetapan LR/epoch terbaik untuk data ini.'),
]


def validate_s3_protocol(bundle, bge, manifest, cfg):
    validate_ridge_inputs(bundle, np.asarray(bge), manifest,
                          dict(cfg, ridge_alphas=cfg['s3_ridge_alphas']))
    if not cfg['s3_ridge_alphas'] or cfg['s2_ridge_alpha'] <= 0:
        raise ValueError('Grid S3 dan alpha kontrol S2 harus positif.')
    if type(cfg['epochs']) is not int or cfg['epochs'] < 1:
        raise ValueError('Jumlah epoch tetap wajib positif.')
    if cfg['representation_policy'] != 'E21_fusion_fixed_epochs_train_only':
        raise ValueError('Notebook ini memakai protokol representasi E21 dengan epoch tetap.')
    y = np.asarray(bundle.y)
    if not np.isfinite(y).all() or not np.isin(y, np.arange(cfg['score_min'], cfg['score_max']+1)).all():
        raise ValueError('Seluruh skor guru harus kategori sah.')


def representation_contract(bundle, bge, cfg, training, inference):
    training, inference = sorted(map(int, training)), sorted(map(int, inference))
    if not training or not inference or len(training) != len(set(training)) or len(inference) != len(set(inference)):
        raise ValueError('Indeks representasi harus unik dan tidak kosong.')
    if not set(training) <= set(inference):
        raise ValueError('Cache fitur harus mencakup train.')
    semantic = {k:v for k,v in cfg.items() if k not in ('dataset','output')}
    return dict(protocol='S3_supervised_representation_train_only_v1',
        source_sha256=bundle.source_sha256, code_sha256=NOTEBOOK_CODE_SHA256,
        training=training, inference=inference,
        train_target_sha256=hashlib.sha256(np.asarray(bundle.y[training],dtype=np.float64).tobytes()).hexdigest(),
        bge_sha256=hashlib.sha256(np.asarray(bge,dtype=np.float32).tobytes()).hexdigest(),
        config=semantic, seed=cfg['seed'], epochs=cfg['epochs'], validation_used_for_training=False,
        torch=str(torch.__version__), device=device_name(cfg),
        gpu=torch.cuda.get_device_name() if device_name(cfg)=='cuda' else None)


def collect_s3_features(model, engine, indices, cfg):
    model.eval()
    arrays = {key:[] for key in ('roberta','roberta_layernorm','projection','concat')}
    predictions = []
    with torch.no_grad():
        for start in range(0,len(indices),cfg['micro_batch']):
            ids=indices[start:start+cfg['micro_batch']]
            p,features=engine.forward(ids,return_features=True)
            for key in arrays:
                arrays[key].append(features[key].detach().float().cpu().numpy())
            predictions.append(p.detach().reshape(-1).float().cpu().numpy() *
                               (cfg['score_max']-cfg['score_min'])+cfg['score_min'])
    arrays={k:np.concatenate(v).astype(np.float32) for k,v in arrays.items()}
    arrays['mlp_prediction']=np.concatenate(predictions).astype(np.float32)
    arrays['indices']=np.asarray(indices,dtype=np.int64)
    if arrays['concat'].shape!=(len(indices),1024) or any(not np.isfinite(v).all() for v in arrays.values()):
        raise ValueError('Fitur/prediksi representasi tidak sah.')
    np.testing.assert_array_equal(arrays['concat'], np.concatenate([arrays['roberta_layernorm'],arrays['projection']],axis=1))
    return arrays


def fit_s3_representation(bundle, bge, cfg, training, inference, cache_root, label, save_final=False):
    """Neither held-out targets nor early stopping can influence this representation."""
    contract=representation_contract(bundle,bge,cfg,training,inference)
    key=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()[:20]
    folder=Path(cache_root)/key;folder.mkdir(parents=True,exist_ok=True)
    complete=folder/'complete.json'; features_path=folder/'features.npz'
    if complete.exists():
        proof=json.loads(complete.read_text())
        if proof['contract']!=contract or digest(features_path)!=proof['features_sha256']:
            raise ValueError('Cache representasi tidak utuh atau kontraknya berbeda; tidak ditimpa.')
        if proof['final_model_saved'] != save_final:
            raise ValueError('Cache final dan cache fold tidak boleh ditukar.')
        if save_final and digest(folder/'final_representation.pt')!=proof['model_sha256']:
            raise ValueError('Checkpoint representasi final berubah.')
        with np.load(features_path,allow_pickle=False) as f: arrays={k:f[k].copy() for k in f.files}
        print(label+': fitur utuh dipakai ulang; training tidak diulang.',flush=True)
        return dict(folder=str(folder),arrays=arrays,proof=proof)
    # Mask all unused targets. Existing fit(..., validation=None) only reads training labels.
    masked=np.full(len(bundle.y),np.nan,dtype=float);masked[training]=bundle.y[training]
    training_bundle=Bundle(rows=bundle.rows,y=masked,groups=bundle.groups,
                           source_sha256=bundle.source_sha256,summary=bundle.summary)
    checkpoint=folder/'epoch_progress.json'
    print(label+f': train={len(training)}, epoch tetap={cfg["epochs"]}; label validasi/test tidak dibaca.',flush=True)
    model,engine,info=fit('S3',training,None,training_bundle,bge,cfg,cfg['epochs'],cfg['seed'],checkpoint=checkpoint)
    info['selection']='Fixed epochs declared before run; no early stopping or held-out labels'
    info['stop_reason']='Predeclared fixed epoch budget completed'
    arrays=collect_s3_features(model,engine,contract['inference'],cfg)
    model_hash=hashlib.sha256()
    for name,t in model.state_dict().items():
        model_hash.update(name.encode());model_hash.update(t.detach().cpu().contiguous().numpy().tobytes())
    model_sha=model_hash.hexdigest()
    if save_final:
        path=folder/'final_representation.pt';temp=path.with_suffix('.pending')
        torch.save(dict(state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
                        contract=contract,info=info),temp);temp.replace(path);model_sha=digest(path)
    temporary=folder/'features.pending.npz';np.savez_compressed(temporary,**arrays);temporary.replace(features_path)
    proof=dict(contract=contract,features_sha256=digest(features_path),model_sha256=model_sha,
               final_model_saved=save_final,info=info,label=label)
    dump(complete,proof)
    # Only this stage's two temporary optimizer snapshots are reclaimed after an intact cache is committed.
    # Interrupted stages retain both snapshots. No older experiment directories are touched.
    if checkpoint.exists():
        progress=json.loads(checkpoint.read_text())
        for snap in progress['snapshots']:
            if Path(snap['file']).name!=snap['file']:raise ValueError('Nama snapshot tidak sah.')
            own_file=folder/snap['file']
            if own_file.is_file():own_file.unlink()
        checkpoint.unlink()
    del engine,model;release()
    return dict(folder=str(folder),arrays=arrays,proof=proof)


def stage_values(stage, indices, key='concat'):
    a=stage['arrays'];lookup={int(v):i for i,v in enumerate(a['indices'])}
    return a[key][[lookup[int(i)] for i in indices]]

In [ ]:
%%aes
def choose_s3_alpha(stages, folds, outer, y, alphas):
    """Inner CV refits the supervised representation too; pair stages are shared symmetrically."""
    search=[]; details=[]
    for alpha in alphas:
        sse=0.;count=0
        for inner in folds:
            if inner['fold']==outer['fold']:continue
            stage=stages[tuple(sorted((outer['fold'],inner['fold'])))]
            tr=stage['proof']['contract']['training']; va=inner['validation']
            forbidden=set(outer['validation'])|set(va)
            if set(tr)&forbidden:raise ValueError('Label inner/outer validation bocor ke representasi.')
            model=Ridge(alpha=float(alpha),fit_intercept=True,solver='cholesky').fit(stage_values(stage,tr).astype(float),y[tr])
            p=model.predict(stage_values(stage,va).astype(float));s=float(np.sum((p-y[va])**2));sse+=s;count+=len(va)
            details.append(dict(outer_fold=outer['fold'],inner_fold=inner['fold'],alpha=float(alpha),
                                train_rows=len(tr),validation_rows=len(va),RMSE=math.sqrt(s/len(va))))
        search.append(dict(stage=f'OUTER_{outer["fold"]}',alpha=float(alpha),selection_RMSE=math.sqrt(sse/count),
                           selection_rows=count,selection_folds=len(folds)-1))
    selected=min(search,key=lambda v:(v['selection_RMSE'],v['alpha']))['alpha']
    for row in search:row['selected']=row['alpha']==selected
    return selected,search,details


def s3_predictions(bundle, indices, p, model, partition, fold, alpha):
    return [dict(model=model,partition=partition,fold=fold,alpha=alpha,row_id=bundle.rows[i]['row_id'],
                 teacher=float(bundle.y[i]),prediction=float(v),error=float(v-bundle.y[i]),
                 absolute_error=float(abs(v-bundle.y[i])),
                 **{f'within{t}':bool(abs(v-bundle.y[i])<=t) for t in (1,2,5,10)}) for i,v in zip(indices,p)]


def s3_evaluation(y,p,cfg,model,partition,fold,alpha):
    return ridge_metric_row(partition,y,p,cfg,model=model,fold=fold,alpha=alpha)

In [ ]:
%%aes
def run_s3_ridge(bundle,bge,manifest,cfg,output):
    X=np.asarray(bge,dtype=np.float32);validate_s3_protocol(bundle,X,manifest,cfg)
    bge_tensor=torch.from_numpy(X);y=np.asarray(bundle.y,dtype=float)
    dev=manifest['development'];test=manifest['test'];folds=manifest['folds']
    run=Path(output);run.mkdir(parents=True,exist_ok=True);cache=run/'representations'
    protocol=dict(experiment='S3_FEATURE_FUSION_RIDGE',code_sha256=NOTEBOOK_CODE_SHA256,
        dataset_sha256=bundle.source_sha256,split_sha256=manifest['split_sha256'],config=cfg,
        representation='E21: RoBERTa CLS768 → LayerNorm; BGE1024 → Linear256 → LayerNorm; concat1024',
        stage_training='S3 MLP auxiliary supervised representation; fixed epochs; train labels only',
        ridge='Post-hoc readout on frozen final fusion features; no gradient through Ridge',
        alpha_grid=cfg['s3_ridge_alphas'],alpha_selection='Nested grouped CV; supervised representation refitted inside inner splits',
        comparison='MLP control uses the exact same fitted fusion features; control head learned jointly during representation training',
        historical_caveat='E21-style architecture; fixed epochs here differ from historical early stopping. Shared historical test is retrospective.',
        unique_representation_fits=len(folds)+len(folds)*(len(folds)-1)//2+1,
        test_used_for_selection=False,BGE_encoder_trained=False)
    dump(run/'protocol_before_test.json',protocol);dump(run/'split_manifest.json',manifest)
    stage_records=[];outer_stages={};pairs={}
    total=protocol['unique_representation_fits'];number=0
    for f in folds:
        number+=1;label=f'Representasi {number}/{total}: outer {f["fold"]}'
        s=fit_s3_representation(bundle,bge_tensor,cfg,f['train'],dev,cache,label)
        outer_stages[f['fold']]=s;stage_records.append(dict(kind='OUTER',fold=f['fold'],folder=s['folder'],proof=s['proof']))
    for a,b in itertools.combinations(folds,2):
        tr=sorted(set(dev)-set(a['validation'])-set(b['validation']));number+=1
        label=f'Representasi {number}/{total}: inner pasangan {a["fold"]}-{b["fold"]}'
        s=fit_s3_representation(bundle,bge_tensor,cfg,tr,dev,cache,label)
        pairs[(a['fold'],b['fold'])]=s;stage_records.append(dict(kind='INNER_PAIR',fold=f'{a["fold"]}-{b["fold"]}',folder=s['folder'],proof=s['proof']))
    evaluations=[];predictions=[];searches=[];inner=[];selected=[]
    oof={model:np.full(len(y),np.nan) for model in S3_MODELS}
    for f in folds:
        tr,va=f['train'],f['validation'];stage=outer_stages[f['fold']]
        alpha,grid,detail=choose_s3_alpha(pairs,folds,f,y,cfg['s3_ridge_alphas'])
        searches.extend(grid);inner.extend(detail);selected.append(dict(fold=f['fold'],alpha=alpha))
        ridge=Ridge(alpha=alpha,fit_intercept=True,solver='cholesky').fit(stage_values(stage,tr).astype(float),y[tr])
        control=Ridge(alpha=cfg['s2_ridge_alpha'],fit_intercept=True,solver='cholesky').fit(X[tr].astype(float),y[tr])
        values={'S3_RIDGE':ridge.predict(stage_values(stage,va).astype(float)),
                'S3_MLP_CONTROL':stage_values(stage,va,'mlp_prediction'),
                'S2_RIDGE_FIXED':control.predict(X[va].astype(float))}
        for model,p in values.items():
            a=alpha if model=='S3_RIDGE' else cfg['s2_ridge_alpha'] if model=='S2_RIDGE_FIXED' else None
            oof[model][va]=p
            predictions.extend(s3_predictions(bundle,va,p,model,'DEVELOPMENT_OOF',f['fold'],a))
            evaluations.append(s3_evaluation(y[va],p,cfg,model,'DEVELOPMENT_OUTER_FOLD',f['fold'],a))
    for model,p in oof.items():
        if not np.isfinite(p[dev]).all() or np.isfinite(p[test]).any():raise ValueError('OOF tidak lengkap atau bercampur test.')
        evaluations.append(s3_evaluation(y[dev],p[dev],cfg,model,'DEVELOPMENT_POOLED_OOF','ALL',None))
    final_search=[]
    for alpha in cfg['s3_ridge_alphas']:
        squared=0.;count=0
        for f in folds:
            tr,va=f['train'],f['validation'];s=outer_stages[f['fold']]
            m=Ridge(alpha=float(alpha),fit_intercept=True,solver='cholesky').fit(stage_values(s,tr).astype(float),y[tr])
            p=m.predict(stage_values(s,va).astype(float));squared+=float(np.sum((p-y[va])**2));count+=len(va)
        final_search.append(dict(stage='FINAL_DEVELOPMENT_SELECTION',alpha=float(alpha),selection_RMSE=math.sqrt(squared/count),
                                 selection_rows=count,selection_folds=len(folds)))
    final_alpha=min(final_search,key=lambda v:(v['selection_RMSE'],v['alpha']))['alpha']
    for row in final_search:row['selected']=row['alpha']==final_alpha
    searches.extend(final_search)
    lock=dict(selected_alpha=final_alpha,selection_partition='development_only',test_used_for_selection=False,
              selection_metric='pooled_RMSE',tie_rule='smaller_alpha',code_sha256=NOTEBOOK_CODE_SHA256,
              split_sha256=manifest['split_sha256'],dataset_sha256=bundle.source_sha256)
    dump(run/'alpha_locked_before_test.json',lock)
    stage=fit_s3_representation(bundle,bge_tensor,cfg,dev,list(range(len(y))),cache,
                                f'Representasi {total}/{total}: final development',save_final=True)
    stage_records.append(dict(kind='FINAL',fold='FINAL',folder=stage['folder'],proof=stage['proof']))
    ridge=Ridge(alpha=final_alpha,fit_intercept=True,solver='cholesky').fit(stage_values(stage,dev).astype(float),y[dev])
    s2=Ridge(alpha=cfg['s2_ridge_alpha'],fit_intercept=True,solver='cholesky').fit(X[dev].astype(float),y[dev])
    for name,model in [('s3_ridge_final.npz',ridge),('s2_ridge_control.npz',s2)]:
        np.savez_compressed(run/name,coef=np.asarray(model.coef_,dtype=float),intercept=np.asarray(model.intercept_,dtype=float),alpha=np.asarray(model.alpha))
    final_features=Path(stage['folder'])/'features.npz'
    final_contract=dict(**lock,representation_checkpoint=str(Path(stage['folder'])/'final_representation.pt'),
        representation_checkpoint_sha256=stage['proof']['model_sha256'],fusion_features_sha256=stage['proof']['features_sha256'],
        BGE=cfg['bge'],RoBERTa=cfg['roberta'],feature_recipe=protocol['representation'],
        s3_ridge_sha256=digest(run/'s3_ridge_final.npz'),s2_ridge_sha256=digest(run/'s2_ridge_control.npz'))
    dump(run/'final_model_contract_before_test.json',final_contract)
    # No test metric or test label was needed to produce either final model.
    for partition,indices in [('TRAIN_FINAL_OPTIMISTIC',dev),('TEST_HISTORICAL_HOLDOUT',test)]:
        values={'S3_RIDGE':ridge.predict(stage_values(stage,indices).astype(float)),
                'S3_MLP_CONTROL':stage_values(stage,indices,'mlp_prediction'),
                'S2_RIDGE_FIXED':s2.predict(X[indices].astype(float))}
        for model,p in values.items():
            a=final_alpha if model=='S3_RIDGE' else cfg['s2_ridge_alpha'] if model=='S2_RIDGE_FIXED' else None
            evaluations.append(s3_evaluation(y[indices],p,cfg,model,partition,'FINAL',a))
            if partition=='TEST_HISTORICAL_HOLDOUT':
                predictions.extend(s3_predictions(bundle,indices,p,model,partition,'FINAL',a))
    result=dict(protocol=protocol,final_contract=final_contract,run_folder=str(run),evaluation=evaluations,
        predictions=predictions,alpha_search=searches,inner_validation=inner,outer_alpha=selected,
        representation_stages=stage_records,final_features=str(final_features),
        final_representation_checkpoint=final_contract['representation_checkpoint'],test_evaluated=True)
    dump(run/'s3_ridge_result.json',result)
    return result

In [ ]:
%%aes
def s3_duels(result):
    rows=[]
    for partition in ('DEVELOPMENT_OOF','TEST_HISTORICAL_HOLDOUT'):
        records=pd.DataFrame([r for r in result['predictions'] if r['partition']==partition])
        for a,b in itertools.combinations(S3_MODELS,2):
            left=records[records.model==a].set_index('row_id');right=records[records.model==b].set_index('row_id').loc[left.index]
            d=left.absolute_error.to_numpy()-right.absolute_error.to_numpy()
            row=dict(partition=partition,left=a,right=b,n=len(d),left_closer=int((d< -1e-8).sum()),
                     right_closer=int((d>1e-8).sum()),ties=int((abs(d)<=1e-8).sum()))
            for t in (1,2,5,10):
                la=left.absolute_error.to_numpy()<=t;ra=right.absolute_error.to_numpy()<=t
                row[f'within{t}_only_left']=int((la&~ra).sum());row[f'within{t}_only_right']=int((ra&~la).sum())
            rows.append(row)
    return rows


def export_s3_ridge(bundle,bge,manifest,cfg,result):
    run=Path(result['run_folder']);out=run/'hasil_excel';out.mkdir(exist_ok=True)
    roles={i:'development' for i in manifest['development']};roles.update({i:'test' for i in manifest['test']})
    data=[dict(row_id=r['row_id'],soal=r['question'],jawaban_referensi=r['reference'],response_text=r['text'],
               rater=float(bundle.y[i]),connected_group=int(bundle.groups[i]),role=roles[i]) for i,r in enumerate(bundle.rows)]
    folds=[dict(fold=f['fold'],row_id=bundle.rows[i]['row_id'],role=role)
           for f in manifest['folds'] for role in ('train','validation') for i in f[role]]
    guide=[dict(Keterangan=k,Nilai=v) for k,v in dict(
        method='S3 feature fusion E21 → Ridge; bukan late fusion dua skor',
        supervision='RoBERTa/norm/proyeksi/MLP belajar train saja; BGE encoder frozen',
        control='MLP kontrol dari checkpoint yang menghasilkan fitur sama; bukan angka lama yang disalin',
        evaluation='Nested OOF untuk S3 Ridge; final alpha dipilih development, test hanya setelah model dikunci',
        epochs='Tetap sebelum run; berbeda dari early stopping E21 historis',
        alpha='Grid pada fitur concat1024 mentah; alpha tidak disamakan skalanya dengan BGE L2',
        limits='Test historis retrospektif; satu seed bukan pengukuhan multi-seed',
        weights='Checkpoint encoder final tetap lokal di path kontrak; NPZ Ridge saja tidak cukup untuk teks baru',
        privacy='Seluruh hasil berisi data privat; jangan commit ke GitHub').items()]
    with np.load(result['final_features'],allow_pickle=False) as f:features={k:f[k].copy() for k in f.files}
    def frame(values,ids):
        df=pd.DataFrame(values,columns=[f'Komponen {i+1}' for i in range(values.shape[1])]);df.insert(0,'row_id',[bundle.rows[int(i)]['row_id'] for i in ids]);return df
    vectors=dict(Panduan=guide,BGE_Frozen=frame(np.asarray(bge),range(len(bundle.rows))))
    for key,title in [('roberta','RoBERTa_Final'),('roberta_layernorm','RoBERTa_LayerNorm'),('projection','BGE_Proyeksi256'),('concat','Fusion1024_Final')]:
        vectors[title]=frame(features[key],features['indices'])
    with np.load(run/'s3_ridge_final.npz',allow_pickle=False) as f:
        vectors['Ridge_Coef']=pd.DataFrame(dict(component=np.arange(1,1025),coefficient=f['coef']))
        vectors['Ridge_Intercept']=[dict(intercept=float(f['intercept']),alpha=float(f['alpha']))]
    write_readout_workbook(out/S3_BOOKS[0],dict(Panduan=guide,Data=data,Lima_Fold=folds))
    write_readout_workbook(out/S3_BOOKS[1],vectors)
    write_readout_workbook(out/S3_BOOKS[2],dict(Panduan=guide,
        Prediksi_Development_OOF=[v for v in result['predictions'] if v['partition']=='DEVELOPMENT_OOF'],
        Prediksi_Test=[v for v in result['predictions'] if v['partition']=='TEST_HISTORICAL_HOLDOUT']))
    write_readout_workbook(out/S3_BOOKS[3],dict(Panduan=guide,Evaluasi=result['evaluation'],Duel=s3_duels(result),
        Alpha_Seleksi=result['alpha_search'],Inner_Validation=result['inner_validation'],Outer_Alpha=result['outer_alpha']))
    history=[];stages=[]
    for s in result['representation_stages']:
        p=s['proof'];c=p['contract'];info=p['info']
        stages.append(dict(kind=s['kind'],fold=s['fold'],train_rows=len(c['training']),inference_rows=len(c['inference']),
            training_indices=c['training'],model_sha256=p['model_sha256'],features_sha256=p['features_sha256'],
            validation_used_for_training=False,parameter_samples_changed=info.get('parameter_samples_changed'),
            optimizer_learning_rates=info.get('optimizer_learning_rates')))
        history.extend(dict(stage_kind=s['kind'],fold=s['fold'],**h) for h in info['history'])
    write_readout_workbook(out/S3_BOOKS[4],dict(Panduan=guide,
        Pengaturan=[dict(parameter=k,value=v) for k,v in dict(cfg,**result['final_contract']).items()],
        Representasi=stages,Riwayat=history,Referensi=S3_REFERENCES,
        Versi=[dict(library=k,version=version(k)) for k in ('torch','transformers','numpy','pandas','scikit-learn','openpyxl')]))
    names={name:out/name for name in S3_BOOKS}
    for name in ('s3_ridge_final.npz','s2_ridge_control.npz','split_manifest.json',
                 'alpha_locked_before_test.json','final_model_contract_before_test.json','protocol_before_test.json'):
        names[name]=run/name
    archive=run/'Hasil_AES_300_S3_Fusion_Ridge_Tiga_Input.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for name,path in names.items():z.write(path,name)
        z.writestr('manifest.json',json.dumps(dict(experiment='S3_FEATURE_FUSION_RIDGE',
            dataset_sha256=bundle.source_sha256,code_sha256=NOTEBOOK_CODE_SHA256,
            split_sha256=manifest['split_sha256'],selected_alpha=result['final_contract']['selected_alpha'],
            test_evaluated=True,test_used_for_selection=False,BGE_encoder_trained=False,
            representation_fits=result['protocol']['unique_representation_fits'],
            full_representation_checkpoint_included=False,
            files={name:digest(path) for name,path in names.items()}),ensure_ascii=False,indent=2))
    return dict(archive=str(archive),books=list(S3_BOOKS),run_folder=str(run),
                final_representation_checkpoint=result['final_representation_checkpoint'],test_evaluated=True)

## 8 · Jalankan S3 Ridge dan kontrol

Satu seed42. Ini bukan ensemble atau bukti kestabilan multi-seed. Head MLP kontrol dilatih bersama representasi; Ridge adalah penggantian readout setelah representasi dikunci. Perbandingan ini belum mengisolasi regularisasi, kelas model dan proses fitting satu per satu.

Rerun dengan data/kode/config/perangkat yang sama memakai ulang tahap lengkap dan melanjutkan epoch yang terputus. Cache yang rusak menghentikan run; tidak diabaikan diam-diam.

In [ ]:
%%aes
result = run_s3_ridge(bundle, X, manifest, cfg, output)
print('S3 alpha final:',result['final_contract']['selected_alpha'],'| Test dievaluasi:',result['test_evaluated'],flush=True)

## 9 · Evaluasi berpasangan

RMSE/MAE/bias dan toleransi memakai prediksi kontinu asli tanpa clipping. QWK/AC2 memakai salinan kategori0–100; nilai undefined diberi alasan. Jumlah dalam±1/±2/±5/±10 dan duel per jawaban dilaporkan agar penurunan RMSE tidak disamakan dengan semua ukuran presisi membaik.

OOF S3 Ridge memakai nested selection, sedangkan nilai grid final hanya untuk memilih alpha dan bukan skor independen. Bandingkan test dengan test pada data yang sama. Jangan menganggap MLP kontrol ini identik dengan run lama yang memakai early stopping.

In [ ]:
%%aes
show(pd.DataFrame([r for r in result['evaluation'] if r['partition']=='DEVELOPMENT_POOLED_OOF']), 'OOF development; S3 Ridge memakai nested selection')
show(pd.DataFrame([r for r in result['evaluation'] if r['partition']=='TEST_HISTORICAL_HOLDOUT']), 'Test historis pada60jawaban jika split240/60terpenuhi')
show(pd.DataFrame(s3_duels(result)), 'Duel pada jawaban sama')
show(pd.DataFrame(result['alpha_search']), 'Seleksi alpha; bukan pemilihan berdasarkan test')
show(pd.DataFrame([r for r in result['predictions'] if r['partition']=='TEST_HISTORICAL_HOLDOUT']), 'Prediksi test per jawaban')

## 10 · Lima Excel dan ZIP privat

Data/split; fitur BGE/RoBERTa/fusion final dan koefisien; prediksi OOF/test; evaluasi/duel/alpha; riwayat semua representasi dan referensi. ZIP juga memuat NPZ Ridge, kontrak model, split dan checksum.

**Checkpoint encoder/proyeksi/MLP final berukuran besar tetap lokal**, path ditampilkan. NPZ Ridge saja tidak cukup untuk menilai teks baru; perlu checkpoint representasi final dan resep token yang sama. Unduh checkpoint final juga jika ingin menyimpan model, sebelum runtime berakhir.

Referensi: [Ridge, Hoerl & Kennard1970](https://doi.org/10.1080/00401706.1970.10488634); [seleksi/evaluasi, Cawley & Talbot2010](https://www.jmlr.org/papers/v11/cawley10a.html); [SentEval2018](https://aclanthology.org/L18-1269/); [BGE-M32024](https://arxiv.org/abs/2402.03216); [RoBERTa2019](https://arxiv.org/abs/1907.11692); [AdamW2019](https://arxiv.org/abs/1711.05101).

Sumber tersebut mendukung komponen/prinsip. Fusi E21, grid alpha, LR dan25epoch merupakan rancangan eksperimen terkontrol; belum ada paper yang diverifikasi membuktikan resep persis ini optimal untuk AES ini.

In [ ]:
%%aes
download_info = export_s3_ridge(bundle, X, manifest, cfg, result)
dump(ROOT/'outputs/s3_ridge_download.json',download_info)
show(pd.DataFrame([download_info]), 'Paket hasil dan path model final')

In [ ]:
DOWNLOAD_FULL_ENCODER = False  # True: unduh checkpoint besar sesudah ZIP hasil.
info=json.loads((ROOT/'outputs/s3_ridge_download.json').read_text())
archive=Path(info['archive']);checkpoint=Path(info['final_representation_checkpoint'])
if not archive.is_file() or not checkpoint.is_file():
    raise FileNotFoundError('Paket/checkpoint final belum lengkap; periksa cell sebelumnya.')
print('ZIP hasil:',archive)
print('Checkpoint representasi final:',checkpoint)
print('NPZ Ridge saja tidak cukup untuk teks baru; simpan checkpoint jika diperlukan.')
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
    if DOWNLOAD_FULL_ENCODER:files.download(str(checkpoint))
else:
    print('Ambil file lokal pada path di atas.')